In [ ]:
# GUMPLE port of SignalBoxSystematics-ReCAF.ipynb (sbn-rewgted-19 dataframes).
# cwd guard + sys.path setup for headless nbconvert (kernel starts in nb/).
import os, sys
if os.path.basename(os.getcwd()) == "nb":
    os.chdir("..")
# mirror the sys.path setup of the gump scripts: the gump dir (kinematics,
# loaddf), ../gumple (gumple_cuts) and the repo root (pyanalib, makedf) --
# headless
# nbconvert has no PYTHONPATH from setup.sh
for _p in (os.getcwd(), os.path.abspath(os.path.join(os.getcwd(), "..", "..")),
           os.path.abspath(os.path.join(os.getcwd(), "..", "gumple"))):
    if _p not in sys.path:
        sys.path.insert(0, _p)
print("cwd:", os.getcwd())

In [ ]:
%load_ext autoreload
%autoreload 2

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import matplotlib as mpl
import h5py

from scipy.interpolate import CubicSpline

from tqdm.auto import tqdm

import warnings
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

from multiprocess import Pool

In [ ]:
import pyanalib.pandas_helpers as ph
from makedf.util import *

import kinematics
import gumple_cuts as gc
import loaddf
import syst

import importlib

In [ ]:
# importlib.reload(gc)

In [ ]:
# Output base directory, overridable by the serial runner / technote driver
# (run_gumple_serial.py, technote/make_technote_plots.sh) via GUMPLE_PLOTBASE.
import os
PLOTBASE = os.environ.get("GUMPLE_PLOTBASE",
                          "/Users/gputnam/Work/osc/cafpyana/plots-gumple-2026-08-26")
PLOTDIR = PLOTBASE + "/signalbox/"

DOSAVE = True

import os
os.makedirs(PLOTDIR, exist_ok=True)
os.makedirs(PLOTDIR + "/png", exist_ok=True)
os.makedirs(PLOTDIR + "/pdf", exist_ok=True)

# Proton-multiplicity selection applied in signalbox():
#   "1p"   -> exactly one proton candidate (n_pfp == 2, the GUMP selection)
#   ">1p"  -> more than one                (n_pfp > 2,  the MAPLE selection)
#   ">=1p" -> at least one                 (n_pfp >= 2, multiplicity-inclusive,
#             the pre-2026-08-25 behavior of this notebook)
PROTON_SEL = "1p"
assert PROTON_SEL in ("1p", ">1p", ">=1p"), PROTON_SEL
print("PROTON_SEL:", PROTON_SEL)

In [ ]:
FONTSIZE = 14
HAWKS_COLORS = ["#315031", "#d54c28", "#1e3f54", "#c89648", "#43140b", "#95af8b"]

def add_style(ax, xlabel, title="", det="ICARUS", ylabel='Events / $10^{20}$ POT', legend_loc=None, legend_ncol=1, legend_title=None, legend_fontsize=FONTSIZE-1):
    ax.tick_params(axis='both', which='both', direction='in', length=6, width=1.5, labelsize=FONTSIZE, top=True, right=True)
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)
    ax.set_xlabel(xlabel, fontsize=FONTSIZE, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=FONTSIZE, fontweight='bold')
    ax.set_title(f"$\\bf{{{det}}}$  {title}", fontsize=FONTSIZE+2)
    ax.legend(fontsize=legend_fontsize, loc=legend_loc, ncol=legend_ncol, title=legend_title, title_fontsize=FONTSIZE)

In [ ]:
RECO = "PANDORA"

In [ ]:
# Input production, overridable via GUMPLE_DF_DIR (see the PID notebook).
DF_DIR = os.environ.get("GUMPLE_DF_DIR", "/Users/gputnam/Work/osc/sbn-rewgted-20/")
if not DF_DIR.endswith("/"):
    DF_DIR += "/"
print("DF_DIR:", DF_DIR)

SCV_FILES = [DF_DIR + "SBNDMCCV_%i.df" % i for i in range(4)]
SDIRT_FILE = DF_DIR + "SBND_SpringLowEMC.df"
SBEAMOFF_FILE = DF_DIR + "SBND_SpringBNBOffData.df"
SDETVARS = [
    ([DF_DIR + "SBND_SpringMC_Nom.df"], [DF_DIR + "SBND_SpringMC_DENT.df"]),
    ([DF_DIR + "SBND_SpringMC_WMNom.df"], [DF_DIR + "SBND_SpringMC_WMXThetaXW.df"]),
    ([DF_DIR + "SBND_SpringMC_WMNom.df"], [DF_DIR + "SBND_SpringMC_WMYZ.df"]),
    # ([DF_DIR + "SBNDMCCV_Nom.df"], [DF_DIR + "SBND_SpringMC_WMXThetaXW.df"]),
    # ([DF_DIR + "SBNDMCCV_Nom.df"], [DF_DIR + "SBND_SpringMC_WMYZ.df"]),
]

SDETVAR_NAMES = ["DENT", "WM $X\\theta_{xw}$", "WM $YZ$"]

IRUN2_CV_FILES = [DF_DIR + "ICARUSRun2_SpringMCOverlay_rewgt_%i.df" % i for i in range(3)]
IRUN4_CV_FILES = [DF_DIR + "ICARUSRun4_SpringMCOverlay_rewgt_%i.df" % i for i in range(4)]
IRUN2_DIRT_FILES = [DF_DIR + "ICARUSRun2_Spring_Overlay_Dirt.df"]
IRUN4_DIRT_FILES = [DF_DIR + "ICARUSRun4_Spring_Overlay_Dirt.df"]
IRUN2_BEAMOFF_FILE = DF_DIR + "ICARUS_SpringRun2BNBOff_unblind.df"
IRUN4_BEAMOFF_FILE = DF_DIR + "ICARUS_SpringRun4BNBOff_unblind.df"

# NB: sbn-rewgted-19 ships no ICARUSRun{2,4}_Spring_Overlay_Nom.df, so the
# detvar CV partner is the rewgt CV sample (same underlying overlay events);
# match_common_evts restricts each pair to the shared events.
IRUN2_DETVARS = [
    (IRUN2_CV_FILES, [DF_DIR + "ICARUSRun2_Spring_Overlay_SCE.df"]),
    (IRUN2_CV_FILES, [DF_DIR + "ICARUSRun2_Spring_Overlay_WMXThXW.df"]),
    (IRUN2_CV_FILES, [DF_DIR + "ICARUSRun2_Spring_Overlay_WMYZ.df"]),
]

IRUN4_DETVARS = [
    (IRUN4_CV_FILES, [DF_DIR + "ICARUSRun4_Spring_Overlay_SCE.df"]),
    (IRUN4_CV_FILES, [DF_DIR + "ICARUSRun4_Spring_Overlay_WMXThXW.df"]),
    (IRUN4_CV_FILES, [DF_DIR + "ICARUSRun4_Spring_Overlay_WMYZ.df"]),
]


IDETVAR_NAMES = [
                 "SCE",
                 "WM $X\\theta_{xw}$",
                 "WM $YZ$",
                ]

IRUN2_POT = 2e20
IRUN4_POT = 3e20

In [ ]:
# IDdf, IDmatch, IDpot = loaddf.load(DF_DIR + "ICARUS_Spring_WMXTHXW_rewgt.df", include_syst=False)

In [ ]:
IGOAL_POT = IRUN2_POT + IRUN4_POT
SGOAL_POT = 1e20

In [ ]:
def FV(df):
    is_spine = "SPINE" in RECO

    # gumple_cuts: cut_contained is the production candidate-containment flag
    # (the old gc.containment_cut analog); slcfv_cut now also excludes the
    # ICARUS WW dangling-cable region.
    ret = gc.slcfv_cut(df) & df.cut_contained

    if is_spine:
        ret = ret & (df.is_time_contained)

    return ret

# NB: the flash cut is NOT applied in the load-time preselection so that the
# trigger (flash-PE scale) systematic can loosen the threshold; it is applied
# in the "selected" definition below instead.
def FVSBND(df):
    return FV(df)

def FVICARUSRun2(df):
    return FV(df)

def FVICARUSRun4(df):
    return FV(df)

In [ ]:
def TrueAV(df):
    vtx = pd.DataFrame({
                        'detector': df.detector,
                        'Run': df.Run,
                        'x': df.true_vtx_x,
                       'y': df.true_vtx_y,
                       'z': df.true_vtx_z}, index=df.index)
    return gc._fv_cut(vtx, 0, 0, 0, 0)

def TrueAVSBND(df):
    return TrueAV(df)

def TrueAVICARUSRun2(df):
    return TrueAV(df)

def TrueAVICARUSRun4(df):
    return TrueAV(df)

def OOAVSBND(df):
    return ~np.isnan(df.true_vtx_x) & ~TrueAVSBND(df)

def OOAVICARUS(df):
    return ~np.isnan(df.true_vtx_x) & ~TrueAVICARUSRun4(df)

In [ ]:
importlib.reload(loaddf)
importlib.reload(gc)
importlib.reload(syst)

In [ ]:
# Load Run 2 and Run 4 ICARUS MC separately
Idf_r2, Imatch_r2, Ipot_r2 = loaddf.loadl(IRUN2_CV_FILES, njob=min(len(IRUN2_CV_FILES), 10), detector="ICARUS Run2", 
                                 preselection=FVICARUSRun2, reweight_aFF=True, pot_univ=True)

Idf_r4, Imatch_r4, Ipot_r4 = loaddf.loadl(IRUN4_CV_FILES, njob=min(len(IRUN4_CV_FILES), 10), detector="ICARUS Run4", 
                                 preselection=FVICARUSRun4, reweight_aFF=True, pot_univ=True)

# Scale each run to its target POT before combining
loaddf.scale_pot(Idf_r2, Ipot_r2, IRUN2_POT)
loaddf.scale_pot(Idf_r4, Ipot_r4, IRUN4_POT)
Idf = pd.concat([Idf_r2, Idf_r4]).reset_index(drop=True)
Imatch = pd.concat([Imatch_r2, Imatch_r4])
Ipot = IRUN2_POT + IRUN4_POT

# Load Run 2 and Run 4 ICARUS dirt separately
Idirt_r2, Idirtmatch_r2, Idirtpot_r2 = loaddf.loadl(IRUN2_DIRT_FILES, njob=min(len(IRUN2_DIRT_FILES), 10), detector="ICARUS Run2", 
                                           preselection=FVICARUSRun2, include_syst=False)
Idirt_r4, Idirtmatch_r4, Idirtpot_r4 = loaddf.loadl(IRUN4_DIRT_FILES, njob=min(len(IRUN4_DIRT_FILES), 10), detector="ICARUS Run4", 
                                           preselection=FVICARUSRun4, include_syst=False)
loaddf.scale_pot(Idirt_r2, Idirtpot_r2, IRUN2_POT)
loaddf.scale_pot(Idirt_r4, Idirtpot_r4, IRUN4_POT)
Idirt = pd.concat([Idirt_r2, Idirt_r4]).reset_index(drop=True)


Idirtpot = IRUN2_POT + IRUN4_POT

# Load Run 2 and Run 4 ICARUS beamoff separately
Ioffbeam_r2, _, Ioffbeampot_r2 = loaddf.load(IRUN2_BEAMOFF_FILE, detector="ICARUS Run2", offbeampot=True, 
                                                   preselection=FVICARUSRun2, include_syst=False, load_truth=False)
Ioffbeam_r4, _, Ioffbeampot_r4 = loaddf.load(IRUN4_BEAMOFF_FILE, detector="ICARUS Run4", offbeampot=True, 
                                                   preselection=FVICARUSRun4, include_syst=False, load_truth=False)
loaddf.scale_pot(Ioffbeam_r2, Ioffbeampot_r2, IRUN2_POT)
loaddf.scale_pot(Ioffbeam_r4, Ioffbeampot_r4, IRUN4_POT)
Ioffbeam = pd.concat([Ioffbeam_r2, Ioffbeam_r4]).reset_index(drop=True)
Ioffbeampot = IRUN2_POT + IRUN4_POT

In [ ]:
Sdf, Smatch, Spot = loaddf.loadl(SCV_FILES, njob=min(len(SCV_FILES), 10), detector="SBND",
                                 preselection=FVSBND, reweight_aFF=True, pot_univ=True)
Sdirt, Sdirtmatch, Sdirtpot = loaddf.load(SDIRT_FILE, detector="SBND",
                                          preselection=FVSBND, include_syst=False)
Soffbeam, Soffbeammatch, Soffbeampot = loaddf.load(SBEAMOFF_FILE, detector="SBND",
                                                   preselection=FVSBND, offbeampot=True, include_syst=False, load_truth=False)

In [ ]:
def load_detvar(detvar, detector):
    if detector == "SBND":
        FV = FVSBND
    elif detector == "ICARUS Run2":
        FV = FVICARUSRun2
    elif detector == "ICARUS Run4":
        FV = FVICARUSRun4
        
    df, match, pots = zip(*[loaddf.loadl(flist, preselection=FV, include_syst=False, detector=detector) for flist in detvar])
    df, pots = loaddf.match_common_evts(match, df, pots)

    return df, pots    

In [ ]:
Sdetvars, Sdetvar_pots = zip(*tqdm([load_detvar(detvar, "SBND") for detvar in SDETVARS]))

In [ ]:
Ir2detvars, Ir2detvar_pots = zip(*tqdm([load_detvar(detvar, "ICARUS Run2") for detvar in IRUN2_DETVARS]))

In [ ]:
Ir4detvars, Ir4detvar_pots = zip(*tqdm([load_detvar(detvar, "ICARUS Run4") for detvar in IRUN4_DETVARS]))

In [ ]:
for c in Sdf.columns:
    if "_univ" in c:
        Sdirt[c] = 1

for c in Idf.columns:
    if "_univ" in c:
        Idirt[c] = 1

if "dirt" not in Sdf.columns:
    Sdf["dirt"] = False
    Sdirt["dirt"] = True

if "dirt" not in Idf.columns:
    Idf["dirt"] = False
    Idirt["dirt"] = True

In [ ]:
for i in range(len(Ir2detvars)):
    for j in range(len(Ir2detvars[i])):
        loaddf.scale_pot(Ir2detvars[i][j], Ir2detvar_pots[i][j], IRUN2_POT)

for i in range(len(Ir4detvars)):
    for j in range(len(Ir4detvars[i])):
        loaddf.scale_pot(Ir4detvars[i][j], Ir4detvar_pots[i][j], IRUN4_POT)

In [ ]:
Idetvars = []

for i in range(len(Ir4detvars)):
    Idetvars.append([])
    for j in range(len(Ir4detvars[i])):
        Idetvars[i].append(pd.concat([Ir2detvars[i][j], Ir4detvars[i][j]]).reset_index(drop=True))

In [ ]:
print("SBND CV")
loaddf.scale_pot(Sdf, Spot, SGOAL_POT)
print("SBND Dirt")
loaddf.scale_pot(Sdirt, Sdirtpot, SGOAL_POT)
print("SBND Beam OFF")
loaddf.scale_pot(Soffbeam, Soffbeampot, SGOAL_POT)
for i in range(len(Sdetvars)):
    print("SBND", SDETVAR_NAMES[i])
    for j in range(len(Sdetvars[i])):
        loaddf.scale_pot(Sdetvars[i][j], Sdetvar_pots[i][j], SGOAL_POT)

In [ ]:
print("ICARUS CV -- already scaled during load")
print("ICARUS Dirt -- already scaled during load")
print("ICARUS Detvars -- already scaled during load")
print("ICARUS Beam OFF -- already scaled during load")

In [ ]:
Sdf = pd.concat([Sdf[~Sdf.dirt], Sdirt])
Idf = pd.concat([Idf[~Idf.dirt], Idirt])

In [ ]:
chi2vars = [
    "mu_chi2_of_mu_cand",
    "prot_chi2_of_mu_cand",
    "mu_chi2_of_prot_cand",
    "prot_chi2_of_prot_cand",
]

chi2labels = [
    "$\\chi^2_\\mu$ of Muon Candidate",
    "$\\chi^2_p$ of Muon Candidate",
    "$\\chi^2_\\mu$ of Proton Candidate",
    "$\\chi^2_p$ of Proton Candidate",
]

# chi2bins = [
#     np.linspace(0, 60, 21),
#     np.linspace(0, 300, 21),
#     np.linspace(0, 60, 21),
#     np.linspace(0, 300, 21)
# ]

chi2bins = [
    np.linspace(0, 60, 21),
    np.linspace(0, 350, 61)[:30],
    np.linspace(0, 60, 21),
    np.linspace(0, 350, 61)[:30]
]

In [ ]:
# Calorimetric (chi2) variations. dedxbias is ICARUS-only: in SBND the
# dedxbias columns are written as a copy of the CV (analysis_village/gump/makedf.py),
# so it would be an identically-zero systematic there.
SCHI2_VARIATIONS      = ["smear13",
                         # "sqsmear15",
                         "hi", "alpha_p", "beta_p", "R_p"]
SCHI2_VARIATION_NAMES = ["dE/dx Constant Res.",
                         # "dE/dx Stochastic Res.",
                         "Gain Hi",
                         "EMB $\\alpha$",
                         "EMB $\\beta$",
                         "EMB $R$"]

ICHI2_VARIATIONS      = ["smear13", "hi", "dedxbias"]
ICHI2_VARIATION_NAMES = ["dE/dx Constant Res.",
                         "Gain Hi",
                         "dE/dx Bias"]

# union of the two, used only to backfill the dirt rows below
chi2_variations = list(dict.fromkeys(SCHI2_VARIATIONS + ICHI2_VARIATIONS))

In [ ]:
# The dirt files carry no chi2-variation columns, and the dirt rows were already
# concatenated into Sdf/Idf above -- so those columns are NaN there. A NaN chi2
# fails gc.pid_cut, which would silently drop the whole dirt sample from every
# chi2 universe while the CV keeps it, inflating each calorimetric systematic by
# roughly the dirt fraction. Fall back to the CV chi2 for those rows, the same
# convention as syst.v_variation.
for c in chi2vars:
    for chi2_var in chi2_variations:
        vc = c.replace("chi2", "chi2" + chi2_var)
        Sdirt[vc] = Sdirt[c]
        Idirt[vc] = Idirt[c]
        Sdf[vc] = Sdf[vc].fillna(Sdf[c]) if vc in Sdf.columns else Sdf[c]
        Idf[vc] = Idf[vc].fillna(Idf[c]) if vc in Idf.columns else Idf[c]

In [ ]:
# Loose selection used only for the area-normalized chi2 shape plots below.
# The FV / containment cuts are already applied at load time by FV().
def select(df):
    return gc.cosmic_cut(df) & gc.pid_cut(df)

In [ ]:
for i, (c,l,b) in enumerate(zip(chi2vars, chi2labels, chi2bins)):
    plt.figure(i)
    _ = plt.hist(Sdf.loc[select(Sdf), c], bins=b, density=True, histtype="step", linewidth=2, label="CV")
    for chi2_var, label in zip(SCHI2_VARIATIONS, SCHI2_VARIATION_NAMES):
        _ = plt.hist(Sdf.loc[select(Sdf), c.replace("chi2", "chi2" + chi2_var)], bins=b, density=True, histtype="step", linewidth=2, label=label)
    add_style(plt.gca(), l, det="SBND", ylabel="Area Normalized")

    if DOSAVE:
        plt.savefig(PLOTDIR + "/png/SBND_%s_variations.png" % c, bbox_inches="tight")
        plt.savefig(PLOTDIR + "/pdf/SBND_%s_variations.pdf" % c, bbox_inches="tight")

In [ ]:
for i, (c,l,b) in enumerate(zip(chi2vars, chi2labels, chi2bins)):
    plt.figure(i)
    _ = plt.hist(Idf[c], bins=b, density=True, histtype="step", linewidth=2, label="CV")
    for chi2_var, label in zip(ICHI2_VARIATIONS, ICHI2_VARIATION_NAMES):
        _ = plt.hist(Idf[c.replace("chi2", "chi2" + chi2_var)], bins=b, density=True, histtype="step", linewidth=2, label=label)
    add_style(plt.gca(), l, det="ICARUS", ylabel="Area Normalized")

    if DOSAVE:
        plt.savefig(PLOTDIR + "/png/ICARUS_%s_variations.png" % c, bbox_inches="tight")
        plt.savefig(PLOTDIR + "/pdf/ICARUS_%s_variations.pdf" % c, bbox_inches="tight")

In [ ]:
ifig = 0
for i, (c,l,b) in enumerate(zip(chi2vars, chi2labels, chi2bins)):
    for j in range(len(Sdetvars)):
        plt.figure(ifig)
        varname = SDETVAR_NAMES[j]
        _ = plt.hist(Sdetvars[j][0].loc[select(Sdetvars[j][0]), c], bins=b, density=True, histtype="step", linewidth=2, label="CV")
        for indd, d in enumerate(Sdetvars[j][1:]):
            _ = plt.hist(d.loc[select(d), c], bins=b, density=True, histtype="step", linewidth=2, label=varname)
        add_style(plt.gca(), l, det="SBND", ylabel="Area Normalized")
    
        if DOSAVE:
            plt.savefig(PLOTDIR + "/png/SBND_detvar_%s_variation_%s.png" % (c, varname.translate(dict.fromkeys(map(ord, "$\\_{}"), None))), bbox_inches="tight")
            plt.savefig(PLOTDIR + "/pdf/SBND_detvar_%s_variation_%s.pdf" % (c, varname.translate(dict.fromkeys(map(ord, "$\\_{}"), None))), bbox_inches="tight")

        ifig += 1

In [ ]:
ifig = 0
for i, (c,l,b) in enumerate(zip(chi2vars, chi2labels, chi2bins)):
    for j in range(len(Sdetvars)):
        plt.figure(ifig)
        varname = SDETVAR_NAMES[j]
        _ = plt.hist(Sdetvars[j][0][c], bins=b, histtype="step", linewidth=2, label="CV", weights=Sdetvars[j][0].glob_scale)
        for indd, d in enumerate(Sdetvars[j][1:]):
            _ = plt.hist(d[c], bins=b, histtype="step", linewidth=2, label=varname, weights=d.glob_scale)
        add_style(plt.gca(), l, det="SBND", ylabel="Events / 1e20 POT")
    
        if DOSAVE:
            plt.savefig(PLOTDIR + "/png/SBND_detvar_absnorm_%s_variation_%s.png" % (c, varname.translate(dict.fromkeys(map(ord, "$\\_{}"), None))), bbox_inches="tight")
            plt.savefig(PLOTDIR + "/pdf/SBND_detvar_absnorm_%s_variation_%s.pdf" % (c, varname.translate(dict.fromkeys(map(ord, "$\\_{}"), None))), bbox_inches="tight")

        ifig += 1

In [ ]:
ifig = 0
for i, (c,l,b) in enumerate(zip(chi2vars, chi2labels, chi2bins)):
    for j in range(len(Idetvars)):
        plt.figure(ifig)
        varname = IDETVAR_NAMES[j]
        _ = plt.hist(Idetvars[j][0][c], bins=b, density=True, histtype="step", linewidth=2, label="CV")
        for indd, d in enumerate(Idetvars[j][1:]):
            _ = plt.hist(d[c], bins=b, density=True, histtype="step", linewidth=2, label=varname)
        add_style(plt.gca(), l, det="ICARUS", ylabel="Area Normalized")
    
        if DOSAVE:
            plt.savefig(PLOTDIR + "/png/ICARUS_detvar_%s_variation_%s.png" % (c, varname), bbox_inches="tight")
            plt.savefig(PLOTDIR + "/pdf/ICARUS_detvar_%s_variation_%s.pdf" % (c, varname), bbox_inches="tight")

        ifig += 1

In [ ]:
# ADD IN PID VARIATIONS
def v_variation(df, setvars):
    df = df[[c for c in df.columns if "univ" not in c]].copy()
    for (new, old) in setvars:
        df[new] = df[old]
    return df

def v_chi2smear(df):
    setvars = [
        ("mu_chi2_of_mu_cand", "mu_chi2smear13_of_mu_cand"),
        ("mu_chi2_of_prot_cand",  "mu_chi2smear13_of_prot_cand"),
        ("prot_chi2_of_mu_cand", "prot_chi2smear13_of_mu_cand"),
        ("prot_chi2_of_prot_cand",  "prot_chi2smear13_of_prot_cand"),
    ]
    return v_variation(df, setvars)


def v_chi2hi(df):
    setvars = [
        ("mu_chi2_of_mu_cand", "mu_chi2hi_of_mu_cand"),
        ("mu_chi2_of_prot_cand",  "mu_chi2hi_of_prot_cand"),
        ("prot_chi2_of_mu_cand", "prot_chi2hi_of_mu_cand"),
        ("prot_chi2_of_prot_cand",  "prot_chi2hi_of_prot_cand"),
    ]
    return v_variation(df, setvars)

def v_calovariation(df, variation):
    setvars = [
        ("mu_chi2_of_mu_cand", "mu_chi2%s_of_mu_cand" % variation),
        ("mu_chi2_of_prot_cand",  "mu_chi2%s_of_prot_cand" % variation),
        ("prot_chi2_of_mu_cand", "prot_chi2%s_of_mu_cand" % variation),
        ("prot_chi2_of_prot_cand",  "prot_chi2%s_of_prot_cand" % variation),
    ]
    return v_variation(df, setvars)

In [ ]:
Schi2_detvars = [v_calovariation(Sdf, V) for V in SCHI2_VARIATIONS]
Ichi2_detvars = [v_calovariation(Idf, V) for V in ICHI2_VARIATIONS]

SCHI2_DETVAR_NAMES = ["dE/dx Constant Res.",
                      # "dE/dx Stochastic Res.",
                      "Calo. Gain", "EMB $\\alpha$", "EMB $\\beta$", "EMB $R$"]
ICHI2_DETVAR_NAMES = ["dE/dx Constant Res.",
                      "Calo. Gain",
                      "dE/dx Bias"]

In [ ]:
S_TRIGGER_NAMES = ["Trigger"]
I_TRIGGER_NAMES = ["Trigger Run 2", "Trigger Run 4"]

In [ ]:
importlib.reload(gc)

In [ ]:
# Signal box = the gumple_cuts.py selection (per-detector gc.SBND_CUTS / gc.ICARUS_CUTS).
# NB: the flash cut is deliberately NOT included here -- it is applied separately in
# the "selected" definitions below so the trigger (flash-PE scale) systematic can vary
# its threshold. This is gc.maple_base_cuts minus the load-time preselection (already
# applied by FV()) and minus the flash cut. gc.cosmic_cut folds in the CRT-hit veto;
# gc.trk_cut = cut_np & has_muon & cut_0shwother; the cathode cut is the stored flag.
# NB no n_pfp == 2 requirement -- as before, the signal box is multiplicity-inclusive
# past the trk cut.
# Proton-multiplicity cut, gated by PROTON_SEL (config cell at the top).
# NB gc.trk_cut already requires n_pfp >= 2 (cut_np), so ">=1p" reproduces
# the multiplicity-inclusive behavior exactly.
_PROTON_SEL_CUTS = {
    "1p":   lambda df: df.n_pfp == 2,
    ">1p":  lambda df: df.n_pfp > 2,
    ">=1p": lambda df: df.n_pfp >= 2,
}

def proton_sel_cut(df):
    return _PROTON_SEL_CUTS[PROTON_SEL](df)

def signalbox(df):
    return gc.cosmic_cut(df) & gc.pid_cut(df) & gc.trk_cut(df) & df.cut_cathode & \
           proton_sel_cut(df)

In [ ]:
assert(np.all(np.isnan(Sdirt[~OOAVSBND(Sdirt)].true_vtx_x)))

In [ ]:
assert(len(Idirt) == 0 or np.all(np.isnan(Idirt[~OOAVICARUS(Idirt)].true_vtx_x)))

In [ ]:
Sdf["selected_noflash"] = signalbox(Sdf)
Sdf["selected"] = Sdf.selected_noflash & gc.flash_cut(Sdf)
Sdirt["selected"] = signalbox(Sdirt) & gc.flash_cut(Sdirt)
Soffbeam["selected"] = signalbox(Soffbeam) & gc.flash_cut(Soffbeam)

for i in range(len(Sdetvars)):
    for j in range(len(Sdetvars[i])):
        Sdetvars[i][j]["selected"] = signalbox(Sdetvars[i][j]) & gc.flash_cut(Sdetvars[i][j])

for i in range(len(Schi2_detvars)):
    Schi2_detvars[i]["selected"] = signalbox(Schi2_detvars[i]) & gc.flash_cut(Schi2_detvars[i])


In [ ]:
Idf["selected_noflash"] = signalbox(Idf)
Idf["selected"] = Idf.selected_noflash & gc.flash_cut(Idf)
if len(Idirt) > 0:
    Idirt["selected"] = signalbox(Idirt) & gc.flash_cut(Idirt)
else:
    Idirt["selected"] = False
Ioffbeam["selected"] = signalbox(Ioffbeam) & gc.flash_cut(Ioffbeam)

for i in range(len(Idetvars)):
    for j in range(len(Idetvars[i])):
        Idetvars[i][j]["selected"] = signalbox(Idetvars[i][j]) & gc.flash_cut(Idetvars[i][j])

for i in range(len(Ichi2_detvars)):
    Ichi2_detvars[i]["selected"] = signalbox(Ichi2_detvars[i]) & gc.flash_cut(Ichi2_detvars[i])


In [ ]:
Sdf["true_E"] = Sdf.true_E.fillna(-1)
Sdirt["true_E"] = Sdirt.true_E.fillna(-1)
Soffbeam["true_E"] = -1

for i in range(len(Sdetvars)):
    for j in range(len(Sdetvars[i])):
        Sdetvars[i][j]["true_E"] = Sdetvars[i][j].true_E.fillna(-1)

for i in range(len(Schi2_detvars)):
    Schi2_detvars[i]["true_E"] = Schi2_detvars[i].true_E.fillna(-1)

In [ ]:
Idf["true_E"] = Idf.true_E.fillna(-1)
Idirt["true_E"] = Idirt.true_E.fillna(-1)
Ioffbeam["true_E"] = -1

for i in range(len(Idetvars)):
    for j in range(len(Idetvars[i])):
        Idetvars[i][j]["true_E"] = Idetvars[i][j].true_E.fillna(-1)

for i in range(len(Ichi2_detvars)):
    Ichi2_detvars[i]["true_E"] = Ichi2_detvars[i].true_E.fillna(-1)

In [ ]:
# ============================================================
# TRIGGER SYSTEMATIC: flash-PE scale-factor uncertainty
# ============================================================
# Best-fit MC PE scale factors from the data/MC fits in FlashMCDataComparison.ipynb
# (applied to flash_maxpe at load time in loaddf.py):
#   SBND: 0.642 +/- 0.005   ICARUS Run2: 0.632 +/- 0.024   ICARUS Run4: 0.358 +/- 0.017
#
# Varying the scale s -> s*(1 -+ f), f = unc/s, is equivalent to varying the
# threshold on flash_maxpe: T -> T/(1 -+ f). The flash cut is applied in the
# "selected" definition (NOT the load-time preselection), so both the
# tightened- and loosened-threshold universes are available.
TRIG_PE_SCALE     = {1: 0.642, 2: 0.632, 4: 0.358}  # Run -> best-fit s (matches loaddf.py)
TRIG_PE_SCALE_UNC = {1: 0.005, 2: 0.024, 4: 0.017}  # Run -> unc. on s
TRIG_THRESHOLD    = {1: 2000, 2: 5000., 4: 1000.}  # Run -> gump_cuts.flash_cut threshold

def trig_flash_cut_var(df, updn, runs=None):
    """Flash cut with the threshold varied by the +/-1 sigma PE-scale uncertainty.

    updn = +1: scale DOWN 1 sigma -> threshold UP   (fewer events)
    updn = -1: scale UP   1 sigma -> threshold DOWN (more events)
    runs: restrict the variation to these runs (others stay at the CV threshold);
          used to make the ICARUS Run2 / Run4 variations uncorrelated.
    """
    sel = pd.Series(False, index=df.index)
    for run, T in TRIG_THRESHOLD.items():
        f = TRIG_PE_SCALE_UNC[run] / TRIG_PE_SCALE[run]
        Tvar = T / (1 - updn*f) if (runs is None or run in runs) else T
        sel = sel | ((df.Run == run) & (df.flash_maxpe > Tvar))
    return sel

# SBND (all rows Run 1). NB: Sdf/Idf include the (MC) dirt rows, which correctly
# participate in the variation; the (data) offbeam samples correctly do not.
Sdf["selected_trig_up"] = Sdf["selected_noflash"] & trig_flash_cut_var(Sdf, +1)
Sdf["selected_trig_dn"] = Sdf["selected_noflash"] & trig_flash_cut_var(Sdf, -1)

# ICARUS: vary one run at a time -> Run2 and Run4 uncorrelated
Idf["selected_trig_r2_up"] = Idf["selected_noflash"] & trig_flash_cut_var(Idf, +1, runs=[2])
Idf["selected_trig_r2_dn"] = Idf["selected_noflash"] & trig_flash_cut_var(Idf, -1, runs=[2])
Idf["selected_trig_r4_up"] = Idf["selected_noflash"] & trig_flash_cut_var(Idf, +1, runs=[4])
Idf["selected_trig_r4_dn"] = Idf["selected_noflash"] & trig_flash_cut_var(Idf, -1, runs=[4])


In [ ]:
# --- Trigger-variation diagnostics ---
trig_cols = {(1, "SBND"): ("selected_trig_up", "selected_trig_dn"),
             (2, "ICARUS"): ("selected_trig_r2_up", "selected_trig_r2_dn"),
             (4, "ICARUS"): ("selected_trig_r4_up", "selected_trig_r4_dn")}

for det, df in [("SBND", Sdf), ("ICARUS", Idf)]:
    for run in sorted(df.Run.unique()):
        upcol, dncol = trig_cols[(run, det)]
        inrun = (df.Run == run)
        T = TRIG_THRESHOLD[run]
        f = TRIG_PE_SCALE_UNC[run] / TRIG_PE_SCALE[run]
        wcv = df.loc[df.selected & inrun, "glob_scale"].sum()
        wup = df.loc[df[upcol] & inrun, "glob_scale"].sum()
        wdn = df.loc[df[dncol] & inrun, "glob_scale"].sum()
        print("%s Run %i: T = %.0f -> %.1f (up) / %.1f (dn), f = %.2f%%" %
              (det, run, T, T/(1-f), T/(1+f), 100*f))
        print("   selected: %.2f -> %.2f (%+.2f%%) up / %.2f (%+.2f%%) dn" %
              (wcv, wup, 100*(wup/wcv - 1), wdn, 100*(wdn/wcv - 1)))

# Consistency: tightening can only remove events, loosening can only add them
assert not (Sdf.selected_trig_up & ~Sdf.selected).any()
assert not (Sdf.selected & ~Sdf.selected_trig_dn).any()
assert not ((Idf.selected_trig_r2_up | Idf.selected_trig_r4_up) & ~Idf.selected).any()
assert not (Idf.selected & ~(Idf.selected_trig_r2_dn & Idf.selected_trig_r4_dn)).any()


In [ ]:
# ============================================================
# TRACK-SPLITTING SYSTEMATIC (ICARUS)
# ============================================================
# Data shows an excess of muon-track endpoints at Z=0 and the cathodes that MC
# does not reproduce: muons crossing those planes are split by reconstruction
# more often in data. Split fractions measured by TrackSplittingCorrection.py
# (see TrackSplittingCorrection.md); the Run2+Run4 combined value per plane is
# applied as a one-sided, symmetrized 1-sigma variation, uncorrelated between
# planes. One systematic per plane, covering both runs together (the east
# cathode only has Run 4 crossers -- it lies outside the Run 2 muon FV).
fv = gc.ICARUSRun4FVCuts
TRACKSPLIT_PLANES = [   # name, dim, coord
    ("Z=0",          "z", 0.0),
    ("East Cathode", "x", 0.5*(fv["C0"]["x"]["min"] + fv["C0"]["x"]["max"])),
    ("West Cathode", "x", 0.5*(fv["C1"]["x"]["min"] + fv["C1"]["x"]["max"])),
]
# Combined Run2+Run4 split fractions, TrackSplittingCorrection-2026-08-21.md
# (the reCAF sbn-rewgted-14 measurement; rerun TrackSplittingCorrection_GUMPLE.py
# on -19 to update)
TRACKSPLIT_FRAC = {"Z=0": 0.1143, "East Cathode": 0.0446, "West Cathode": 0.0314}

Isplit_systs = []
I_TRACKSPLIT_NAMES = []
for name, dim, coord in TRACKSPLIT_PLANES:
    sdf, crosses = syst.split_tracks(Idf, dim, coord, runs=[2, 4])
    # re-evaluate the selection with the truncated muon kinematics
    sdf["selected"] = signalbox(sdf) & gc.flash_cut(sdf)
    Isplit_systs.append(syst.TrackSplittingSystematic(Idf, sdf, crosses, TRACKSPLIT_FRAC[name]))
    I_TRACKSPLIT_NAMES.append("Trk. Split %s" % name)


In [ ]:
# --- Track-splitting diagnostics ---
for name, s in zip(I_TRACKSPLIT_NAMES, Isplit_systs):
    wcv = Idf.loc[Idf.selected, "glob_scale"].sum()
    w = Idf["glob_scale"].to_numpy()*(1 - s.frac*s.crosses)
    wuniv = w[Idf.selected.to_numpy()].sum() + s.frac*s.splitdf.loc[s.splitdf.selected, "glob_scale"].sum()
    print("%-26s: f = %5.2f%%, %5i crossing muons (%4i still selected after split), selected %.2f -> %.2f (%+.3f%%)"
          % (name, 100*s.frac, s.crosses.sum(), int(s.splitdf.selected.sum()), wcv, wuniv, 100*(wuniv/wcv - 1)))


In [ ]:
# ============================================================
# BINDING-ENERGY SYSTEMATIC (SBND + ICARUS)
# ============================================================
# +25 MeV shift of the binding energy in the reco neutrino-energy formula
# (kinematics.BE = 29.5 MeV), built as a shifted-CV universe df and treated as
# an additional cross-section systematic, fully correlated between SBND and
# ICARUS: a single one-sided universe, symmetrized (like the single-universe
# xsec morphs). The shift is per interaction mode: nominal for QE/RES/DIS,
# x sqrt(2) for MEC (two-nucleon initial state), zero for COH / non-neutrino
# rows. The selection cuts use none of the recomputed columns, so 'selected'
# carries over from the CV; only the columns the covariance machinery reads
# (+ the recompute inputs) are kept, since the CV dfs carry ~200 weight
# columns that a full copy would duplicate.
BE_SHIFT = 0.025
# NB: syst.recompute_kinematics is migrated to the GUMPLE psum convention --
# it requires the stored psum_E / psum_dir_* (and n_pfp) and raises without them.
BE_COLS = ["selected", "glob_scale", "genie_mode", "mu_E", "p_E",
           "mu_dir_x", "mu_dir_y", "mu_dir_z", "p_dir_x", "p_dir_y", "p_dir_z",
           "psum_E", "psum_p", "psum_dir_x", "psum_dir_y", "psum_dir_z", "n_pfp",
           # stored kinematics: the unshifted rows of the in-place BE universe
           # keep these CV values verbatim (nu_E_ccqe is NOT stored -- the
           # universe rebuilds it at nominal BE for the unshifted rows)
           "nu_E_calo", "del_p", "del_Tp", "del_phi"]
Sbedf = syst.shift_binding_energy(Sdf[BE_COLS], BE_SHIFT, fraction=0.5)
Ibedf = syst.shift_binding_energy(Idf[BE_COLS], BE_SHIFT, fraction=0.5)
S_BE_syst = syst.SampleSystematic(Sbedf)
I_BE_syst = syst.SampleSystematic(Ibedf)

In [ ]:
# XSec: the weight-based knobs plus the binding-energy shift (previous cell),
# grouped in a SystematicList so the "XSec" label covers both.
SXSecWgtSyst = loaddf.XSecSystematic(Sdf)
IXSecWgtSyst = loaddf.XSecSystematic(Idf)

Ssystematics = [
    loaddf.FluxSystematic(Sdf),
    loaddf.G4Systematic(Sdf),
    syst.SystematicList([SXSecWgtSyst, S_BE_syst]),
    syst.NormalizationSystematic(0.005),
    syst.SystematicList(
        [syst.SampleSystematic(detvar[1:], cvdf=detvar[0]) for detvar in Sdetvars] + # detector variation samples
        [syst.SampleSystematic(d, norm=(2. if n == "smear13" else 1.)) for d,n in zip(Schi2_detvars, SCHI2_VARIATIONS)] + # chi2 variations
        [syst.SelectionSystematic(Sdf, ["selected_trig_up", "selected_trig_dn"])] + # Trigger (flash-PE scale)
        [syst.SystSampleSystematic(Sdf[Sdf.true_iscosmic], norm=0.107)] + # Cosmic normalization from the off-beam / intime cosmic MC comparison (OffBeamCosmicMCComparison.ipynb)
        [syst.StatSampleSystematic(Sdf)] # MC Stat
    ),
    syst.SystSampleSystematic(Sdf[OOAVSBND(Sdf)]),
    syst.StatSampleSystematic(Soffbeam),
]

Isystematics = [
    loaddf.FluxSystematic(Idf),
    loaddf.G4Systematic(Idf),
    syst.SystematicList([IXSecWgtSyst, I_BE_syst]),
    syst.NormalizationSystematic(0.005),
    syst.SystematicList(
        [syst.SampleSystematic(detvar[1:], cvdf=detvar[0]) for detvar in Idetvars] + # detector variation samples
        [syst.SampleSystematic(d) for d in Ichi2_detvars] + # chi2 variations
        # Trigger (flash-PE scale): Run2 and Run4 varied independently -> uncorrelated sum
        [
            syst.SelectionSystematic(Idf, ["selected_trig_r2_up", "selected_trig_r2_dn"]),
            syst.SelectionSystematic(Idf, ["selected_trig_r4_up", "selected_trig_r4_dn"])
        ] +
        Isplit_systs + # Track splitting at Z=0 / cathodes, per plane x run
        [syst.StatSampleSystematic(Idf)] # MC Stat
    ),
    syst.SystSampleSystematic(Idf[OOAVICARUS(Idf)]),
    syst.StatSampleSystematic(Ioffbeam, norm=0.5), # TODO: change after unblinding. Simulate scaling up stats by 2x.

]

systematics = [
    syst.CorrelatedSystematic(Ssystematics[0], Isystematics[0]),
    syst.CorrelatedSystematic(Ssystematics[1], Isystematics[1]),
    # XSec: each piece (weight knobs, BE shift) fully correlated SBND<->ICARUS
    syst.SystematicList(
        [
            syst.CorrelatedSystematic(SXSecWgtSyst, IXSecWgtSyst),
            syst.CorrelatedSystematic(S_BE_syst, I_BE_syst),
        ]),
    # POT norm is correlated SBND Run1 (1e20) - ICARUS Run4 (3e20), and uncorrelated ICARUS Run 2 (2e20)
    syst.SystematicList(
        [
            syst.UnCorrelatedSystematic(syst.NormalizationSystematic(0), syst.NormalizationSystematic(0.005*(2./5.))),
            syst.CorrelatedSystematic(syst.NormalizationSystematic(0.005), syst.NormalizationSystematic(0.005*(3./5.))),
        ]),
    syst.UnCorrelatedSystematic(Ssystematics[4], Isystematics[4]),
    syst.UnCorrelatedSystematic(Ssystematics[5], Isystematics[5]),
    syst.UnCorrelatedSystematic(Ssystematics[6], Isystematics[6]),
]


labels = [
    "Flux",
    "G4",
    "XSec",
    "POT Norm.",
    "Detector",
    "Dirt",
    "Beam Off",
    "Stat",
]


In [ ]:
# # Impact of CVwgt
# _ = plt.hist(Sdf.loc[Sdf[cut], "true_E"], weights=Sdf.loc[Sdf[cut], "glob_scale"], 
#              bins=np.linspace(0, 2, 11), histtype="step", linewidth=2)
# _ = plt.hist(Sdf.loc[Sdf[cut], "true_E"], weights=Sdf.loc[Sdf[cut], "glob_scale"]/Sdf.loc[Sdf[cut], "cvwgt"], 
#              bins=np.linspace(0, 2, 11), histtype="step", linewidth=2)

In [ ]:
# # Impact of CVwgt
# _ = plt.hist(Idf.loc[Idf[cut], "true_E"], weights=Idf.loc[Idf[cut], "glob_scale"], 
#              bins=np.linspace(0, 2, 11), histtype="step", linewidth=2)
# _ = plt.hist(Idf.loc[Idf[cut], "true_E"], weights=Idf.loc[Idf[cut], "glob_scale"]/Idf.loc[Idf[cut], "cvwgt"], 
#              bins=np.linspace(0, 2, 11), histtype="step", linewidth=2)

In [ ]:
var = "nu_E_calo"
wgt = "glob_scale"
cut = "selected"
# bins = np.linspace(0, 1.5, 11)[2:]
# fine binning (restored 2026-09-02; the coarse 0.0/0.4..1.0/1.5 scheme with
# catch-all edge bins was used 2026-08-25 .. 2026-09-02):
bins = np.array([0.3, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0, 1.25, 1.5])
# coarse binning:
# bins = np.array([0.0, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.5])
centers = (bins[1:] + bins[:-1]) / 2

# With the coarse scheme the lo (0.0-0.4) and hi (1.0-1.5) bins are catch-all
# under/overflow bins and are dropped from the plotted reco-energy uncertainty
# budgets (the same [1:-1] convention the trueE plots use for their real
# under/overflow bins). The fine scheme has no catch-all bins, so nothing is
# dropped. The covariances keep every bin either way.
DROP_EDGE_BINS = bins[0] == 0.0
EDGE = slice(1, -1) if DROP_EDGE_BINS else slice(None)
print("reco bins:", bins, " drop edge bins:", DROP_EDGE_BINS)

def hist_dropedges(centers, bins, weights, **kw):
    return plt.hist(np.asarray(centers)[EDGE], bins=np.asarray(bins)[EDGE],
                    weights=np.asarray(weights)[EDGE], **kw)

In [ ]:
# lo/hi catch-all bins dropped (bins[EDGE]), as in the budget plots
_ = plt.hist(Idf.loc[Idf[cut] & ~OOAVICARUS(Idf), var], bins=bins[EDGE], weights=Idf.loc[Idf[cut] & ~OOAVICARUS(Idf), wgt], 
             histtype="step", linewidth=2, label="CV")
_ = plt.hist(Idf.loc[Idf[cut] & OOAVICARUS(Idf), var], bins=bins[EDGE], weights=Idf.loc[Idf[cut] & OOAVICARUS(Idf), wgt], 
             histtype="step", linewidth=2, label="OOAV")
_ = plt.hist(Ioffbeam.loc[Ioffbeam[cut], var], bins=bins[EDGE], weights=Ioffbeam.loc[Ioffbeam[cut], wgt], histtype="step", linewidth=2, label="Beam Off")

add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="ICARUS")
plt.yscale("log")

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/ICARUS_altsamples.png", bbox_inches="tight")
    plt.savefig(PLOTDIR + "/pdf/ICARUS_altsamples.pdf", bbox_inches="tight")
    

In [ ]:
# lo/hi catch-all bins dropped (bins[EDGE]), as in the budget plots
_ = plt.hist(Sdf.loc[Sdf[cut] & ~OOAVSBND(Sdf), var], bins=bins[EDGE], weights=Sdf.loc[Sdf[cut] & ~OOAVSBND(Sdf), wgt], 
             histtype="step", linewidth=2, label="CV")
_ = plt.hist(Sdf.loc[Sdf[cut] & OOAVSBND(Sdf), var], bins=bins[EDGE], weights=Sdf.loc[Sdf[cut] & OOAVSBND(Sdf), wgt], 
             histtype="step", linewidth=2, label="OOAV")
# _ = plt.hist(Sdirt.loc[Sdirt[cut], var], bins=bins[EDGE], weights=Sdirt.loc[Sdirt[cut], wgt], histtype="step", linewidth=2, label="Dirt")
_ = plt.hist(Soffbeam.loc[Soffbeam[cut], var], bins=bins[EDGE], weights=Soffbeam.loc[Soffbeam[cut], wgt], histtype="step", linewidth=2, label="Beam Off")

add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="SBND")
plt.yscale("log")

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/SBND_altsamples.png", bbox_inches="tight")
    plt.savefig(PLOTDIR + "/pdf/SBND_altsamples.pdf", bbox_inches="tight")
    

In [ ]:
_ = plt.hist(Sdf.loc[Sdf[cut], var], bins=bins, weights=Sdf.loc[Sdf[cut], wgt], histtype="step", linewidth=2, label="CV")

for i in range(20):
    w = Sdf[wgt]*Sdf["flux_univ%i" % i]
    label = "Flux Univ's" if i == 0 else None
    _ = plt.hist(Sdf.loc[Sdf[cut], var], bins=bins, weights=w[Sdf[cut]], histtype="step", linewidth=1, color="gray", label=label)


add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="SBND")

In [ ]:
_ = plt.hist(Sdf.loc[Sdf[cut], var], bins=bins, weights=Sdf.loc[Sdf[cut], wgt], histtype="step", linewidth=2, label="CV")

for i in range(20):
    w = Sdf[wgt]*Sdf["g4_univ%i" % i]
    label = "G4 Univ's" if i == 0 else None
    _ = plt.hist(Sdf.loc[Sdf[cut], var], bins=bins, weights=w[Sdf[cut]], histtype="step", linewidth=1, color="gray", label=label)


add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="SBND")

In [ ]:
_ = plt.hist(Idf.loc[Idf[cut], var], bins=bins, weights=Idf.loc[Idf[cut], wgt], histtype="step", linewidth=2, label="CV")

for i in range(20):
    w = Idf[wgt]*Idf["flux_univ%i" % i]
    label = "Flux Univ's" if i == 0 else None
    _ = plt.hist(Idf.loc[Idf[cut], var], bins=bins, weights=w[Idf[cut]], histtype="step", linewidth=1, color="gray", label=label)

add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="ICARUS")

In [ ]:
_ = plt.hist(Idf.loc[Idf[cut], var], bins=bins, weights=Idf.loc[Idf[cut], wgt], histtype="step", linewidth=2, label="CV")

for i in range(20):
    w = Idf[wgt]*Idf["g4_univ%i" % i]
    label = "G4 Univ's" if i == 0 else None
    _ = plt.hist(Idf.loc[Idf[cut], var], bins=bins, weights=w[Idf[cut]], histtype="step", linewidth=1, color="gray", label=label)

add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="ICARUS")

In [ ]:
_ = plt.hist(Sdf.loc[Sdf[cut], var], bins=bins, weights=Sdf.loc[Sdf[cut], wgt], histtype="step", linewidth=2, label="CV")

for i,s in enumerate(loaddf.xsec_syst):
    w = Sdf[wgt]*Sdf["%s_univ" % s]
    label = "XSec Syst's" if i == 0 else None
    _ = plt.hist(Sdf.loc[Sdf[cut], var], bins=bins, weights=w[Sdf[cut]], histtype="step", linewidth=1, color="gray", label=label)


add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="SBND")

In [ ]:
xsec_systs_toplot = loaddf.xsec_syst

In [ ]:
CV = np.histogram(Sdf.loc[Sdf[cut], var], bins=bins, weights=Sdf.loc[Sdf[cut], wgt])[0]


vs = []
ls = []

for i,s in enumerate(xsec_systs_toplot):
    w = Sdf[wgt]*Sdf["%s_univ" % s]
    v = np.histogram(Sdf.loc[Sdf[cut], var], bins=bins, weights=w[Sdf[cut]])[0]
    _ = plt.hist(centers, bins=bins, weights=v/CV, histtype="step", linewidth=2)

plt.ylim([0.8, 1.2])
plt.axhline([1], color="gray", linestyle="--")


# add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="SBND")

In [ ]:
for s in xsec_systs_toplot:
    if np.any(np.isnan(Sdf["%s_univ" % s])):
        print(s, np.isnan(Sdf["%s_univ" % s]).sum())

In [ ]:
CV = np.histogram(Idf.loc[Idf[cut], var], bins=bins, weights=Idf.loc[Idf[cut], wgt])[0]

for i,s in enumerate(xsec_systs_toplot):
    w = Idf[wgt]*Idf["%s_univ" % s]
    v = np.histogram(Idf.loc[Idf[cut], var], bins=bins, weights=w[Idf[cut]])[0]
    _ = plt.hist(centers, bins=bins, weights=v/CV, histtype="step", linewidth=2)

plt.ylim([0.8, 1.2])
plt.axhline([1], color="gray", linestyle="--")


# add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="SBND")

In [ ]:
_ = plt.hist(Idf.loc[Idf[cut], var], bins=bins, weights=Idf.loc[Idf[cut], wgt], histtype="step", linewidth=2, label="CV")

for i,s in enumerate(loaddf.xsec_syst):
    w = Idf[wgt]*Idf["%s_univ" % s]
    label = "XSec Syts's" if i == 0 else None
    _ = plt.hist(Idf.loc[Idf[cut], var], bins=bins, weights=w[Idf[cut]], histtype="step", linewidth=1, color="gray", label=label)


add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="ICARUS")

In [ ]:
def ratioplot_varyselect(plt, df, selects, names, title, scales=None):
    dfs = []
    for select in selects:
        dfs.append(df[[cut, var, "glob_scale"]].copy())
        dfs[-1][cut] = df[select]

    return ratioplot(plt, dfs, names, title, scales=scales)

def ratioplot(plt, detvars, names, title, presel=None, scales=None, ratio_range=0.1):
    fig, (ax_top, ax_bot) = plt.subplots(
      2, 1, sharex=True,
      gridspec_kw={"height_ratios": [3, 1], "hspace": 0},
    )
    
    centers = 0.5 * (bins[1:] + bins[:-1])
    
    cv_vals = None
    for i in range(len(detvars)):
        d = detvars[i]
        c = d[cut]
        if presel is not None:
            c = c & presel(d)
        
        n, _ = np.histogram(d.loc[c, var], bins=bins,
                          weights=d.loc[c, wgt])
        s = scales[i] if scales is not None else 1.
        n = n * s
        # drop the lo/hi catch-all bins from the drawing (same [EDGE]
        # convention as the budget and trueE plots); the histogramming and the
        # ratio keep every bin
        _ = ax_top.hist(centers[EDGE], bins=bins[EDGE], weights=n[EDGE], color="tab:gray" if i == 0 else None,
                  histtype="step", linewidth=2, label=names[i])
        if i == 0:
            cv_vals = n
        else:
            ratio = np.divide(n, cv_vals, out=np.ones_like(n, dtype=float),
                            where=cv_vals > 0)
            _ = ax_bot.hist(centers[EDGE], bins=bins[EDGE], weights=ratio[EDGE], 
                      histtype="step", linewidth=2)
    
    _ = ax_bot.axhline(1, color="gray", linestyle="--", linewidth=1)
    
    _ = add_style(ax_top, "", det=title)
    _ = ax_top.set_xlabel("")
    
    _ = ax_bot.set_xlabel("Reco. Neutrino Energy [GeV]", fontsize=FONTSIZE, fontweight="bold")
    _ = ax_bot.set_ylabel("Var. / Nom.", fontsize=FONTSIZE, fontweight="bold")
    _ = ax_bot.tick_params(axis="both", which="both", direction="in",
                     length=6, width=1.5, labelsize=FONTSIZE, top=True, right=True)
    _ = ax_bot.set_ylim([1-ratio_range/2., 1+ratio_range/2.])
    # _ = ax_bot.set_ylim([0.95, 1.05])
    for spine in ax_bot.spines.values():
        spine.set_linewidth(1.5)

    return ax_top, ax_bot

In [ ]:
shifts = [0.005, 0.01, 0.025]

be_shifted = [syst.shift_binding_energy(Sdf[BE_COLS], s, fraction=1) for s in shifts]

be_shifted += [syst.shift_binding_energy(Sdf[BE_COLS], shifts[-1], fraction=0.5)]

top, bot = ratioplot(plt, [Sdf] + be_shifted, 
                     ["CV"] + ["$\\Delta E_\\mathrm{miss}$: %i MeV" % (s*1000) for s in shifts] +
                     ["50%% $\\Delta E_\\mathrm{miss}$: %i MeV" % (shifts[-1]*1000)], 
                     "SBND", 
                     ratio_range=1)
top.set_title("SBND Shifted $E_\\mathrm{miss}$")

In [ ]:
shifts = [0.005, 0.01, 0.025]

be_shifted = [syst.shift_binding_energy(Idf[BE_COLS], s, fraction=1) for s in shifts]

be_shifted += [syst.shift_binding_energy(Idf[BE_COLS], shifts[-1], fraction=0.5)]

top, bot = ratioplot(plt, [Idf] + be_shifted, 
                     ["CV"] + ["$\\Delta E_\\mathrm{miss}$: %i MeV" % (s*1000) for s in shifts] +
                     ["50%% $\\Delta E_\\mathrm{miss}$: %i MeV" % (shifts[-1]*1000)], 
                     "ICARUS", 
                     ratio_range=1)
top.set_title("ICARUS Shifted $E_\\mathrm{miss}$")

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/ICARUS_be_shift_variations.png", bbox_inches="tight")
    plt.savefig(PLOTDIR + "/pdf/ICARUS_be_shift_variations.pdf", bbox_inches="tight")

In [ ]:
for i, (name, detvar) in enumerate(zip(SDETVAR_NAMES, Sdetvars)):
    varnames = ["CV", "$+1\\sigma$", "$-1\\sigma$"]
    plt.figure(i)
    top, bot = ratioplot(plt, detvar, varnames, "SBND")
    top.set_title("SBND %s" % name, fontsize=14)

In [ ]:
for i, (name, detvar) in enumerate(zip(IDETVAR_NAMES, Idetvars)):
    varnames = ["CV", "$+1\\sigma$", "$-1\\sigma$"]
    plt.figure(i)
    top, bot = ratioplot(plt, detvar, varnames, "ICARUS")
    top.set_title("ICARUS %s" % name, fontsize=14)

In [ ]:
for i, (name, detvar) in enumerate(zip(IDETVAR_NAMES, Idetvars)):
    varnames = ["CV", "$+1\\sigma$", "$-1\\sigma$"]
    plt.figure(i)
    top, bot = ratioplot(plt, [d[d.Run == 2] for d in detvar], varnames, "ICARUS Run2")
    top.set_title("ICARUS Run2 %s" % name, fontsize=14)

In [ ]:
for i, (name, detvar) in enumerate(zip(IDETVAR_NAMES, Idetvars)):
    varnames = ["CV", "$+1\\sigma$", "$-1\\sigma$"]
    plt.figure(i)
    top, bot = ratioplot(plt, [d[d.Run == 4] for d in detvar], varnames, "ICARUS Run4")
    top.set_title("ICARUS Run4 %s" % name, fontsize=14)

In [ ]:
varnames = ["CV", "$+1\\sigma$", "$-1\\sigma$"]
selects = ["selected", "selected_trig_up", "selected_trig_dn"]

plt.figure(i)
top, bot = ratioplot_varyselect(plt, Sdf, selects, varnames, "SBND")
top.set_title("SBND Trigger", fontsize=14)

In [ ]:
varnames = ["CV", "$+1\\sigma$", "$-1\\sigma$"]
selects = ["selected", "selected_trig_r2_up", "selected_trig_r2_dn"]

plt.figure(i)
top, bot = ratioplot_varyselect(plt, Idf[Idf.Run == 2], selects, varnames, "ICARUS Run2")
top.set_title("ICARUS Run 2 Trigger", fontsize=14)

In [ ]:
varnames = ["CV", "$+1\\sigma$", "$-1\\sigma$"]
selects = ["selected", "selected_trig_r4_up", "selected_trig_r2_dn"]

plt.figure(i)
top, bot = ratioplot_varyselect(plt, Idf[Idf.Run == 4], selects, varnames, "ICARUS Run4")
top.set_title("ICARUS Run 4 Trigger", fontsize=14)

In [ ]:
def tracksplit_universe_df(s):
    """The split universe as a plain df: (1-f) x nominal + f x split rows."""
    base = Idf[[cut, var, wgt]].copy()
    base[wgt] = Idf[wgt].to_numpy()*(1 - s.frac*s.crosses)
    sp = s.splitdf[[cut, var, wgt]].copy()
    sp[wgt] = sp[wgt]*s.frac
    return pd.concat([base, sp])

for i, (name, s) in enumerate(zip(I_TRACKSPLIT_NAMES, Isplit_systs)):
    plt.figure(i)
    top, bot = ratioplot(plt, [Idf, tracksplit_universe_df(s)], ["CV", "Split Universe"], "ICARUS")
    top.set_title("ICARUS %s" % name, fontsize=14)


In [ ]:
# for i in range(len(Sdetvars)):
#     _ = plt.hist(Sdetvars[i].loc[Sdetvars[i][cut], var], bins=bins, weights=Sdetvars[i].loc[Sdetvars[i][cut], wgt], 
#             histtype="step", linewidth=2, label=SDETVAR_NAMES[i])

# add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="SBND")

In [ ]:
_ = plt.hist(Sdf.loc[Sdf[cut], var], bins=bins, weights=Sdf.loc[Sdf[cut], wgt], histtype="step", linewidth=2, label="Nominal")

for i in range(len(Schi2_detvars)):
    _ = plt.hist(Schi2_detvars[i].loc[Schi2_detvars[i][cut], var], bins=bins, weights=Schi2_detvars[i].loc[Schi2_detvars[i][cut], wgt], 
            histtype="step", linewidth=2, label=SCHI2_DETVAR_NAMES[i])

add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="SBND")

In [ ]:
_ = plt.hist(Idf.loc[Idf[cut], var], bins=bins, weights=Idf.loc[Idf[cut], wgt], histtype="step", linewidth=2, label="CV")

for i in range(len(Ichi2_detvars)):
    _ = plt.hist(Ichi2_detvars[i].loc[Ichi2_detvars[i][cut], var], bins=bins, weights=Ichi2_detvars[i].loc[Ichi2_detvars[i][cut], wgt], 
            histtype="step", linewidth=2, label=ICHI2_DETVAR_NAMES[i])

add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="ICARUS")

In [ ]:
SCV = np.histogram(Sdf.loc[Sdf[cut], var], bins=bins, weights=Sdf.loc[Sdf[cut], wgt])[0]
Scovs = [s.cov(var, cut, bins, SCV) for s in Ssystematics]
Scovs.append(np.diag(SCV))

In [ ]:
ICV = np.histogram(Idf.loc[Idf[cut], var], bins=bins, weights=Idf.loc[Idf[cut], wgt])[0]
Icovs = [s.cov(var, cut, bins, ICV) for s in Isystematics]
Icovs.append(np.diag(ICV))

In [ ]:
S_detsyst_covs = [s.cov(var, cut, bins, SCV) for s in Ssystematics[labels.index("Detector")].systs]
I_detsyst_covs = [s.cov(var, cut, bins, ICV) for s in Isystematics[labels.index("Detector")].systs]

In [ ]:
for c, l in zip(Scovs, labels):
    _ = hist_dropedges(centers, bins, (np.sqrt(np.diag(c))/SCV), label=l, histtype="step", linewidth=2)

add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="SBND", ylabel="Fractional Uncertainty", 
          legend_loc="upper center", legend_ncol=2)

plt.ylim([0, 0.5])

# plt.yscale("log")

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/SBND_signalbox_systematics.png", bbox_inches="tight")
    plt.savefig(PLOTDIR + "/pdf/SBND_signalbox_systematics.pdf", bbox_inches="tight")

In [ ]:
for c, l in zip(Icovs, labels):
    _ = hist_dropedges(centers, bins, (np.sqrt(np.diag(c))/ICV), label=l, histtype="step", linewidth=2)

add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="ICARUS", ylabel="Fractional Uncertainty", 
          legend_loc="upper center", legend_ncol=2)

plt.ylim([0, 0.5])

# plt.yscale("log")

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/ICARUS_signalbox_systematics.png", bbox_inches="tight")
    plt.savefig(PLOTDIR + "/pdf/ICARUS_signalbox_systematics.pdf", bbox_inches="tight")

In [ ]:
for c, l in zip(Scovs, labels):
    _ = hist_dropedges(centers, bins, (np.sqrt(np.diag(c))/SCV), label=l, histtype="step", linewidth=2)

add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="SBND", ylabel="Fractional Uncertainty", 
          legend_loc="upper center", legend_ncol=2)

plt.ylim([1e-3, 8])

plt.yscale("log")

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/SBND_signalbox_systematics_log.png", bbox_inches="tight")
    plt.savefig(PLOTDIR + "/pdf/SBND_signalbox_systematics_log.pdf", bbox_inches="tight")

In [ ]:
combined = np.zeros(Scovs[0].shape)
for c, l in zip(S_detsyst_covs, SDETVAR_NAMES + SCHI2_DETVAR_NAMES + S_TRIGGER_NAMES + ["Corsika", "MC Stat"]):
    linestyle = "--" if l == "MC Stat" else "-"
    _ = hist_dropedges(centers, bins, (np.sqrt(np.diag(c))/SCV), label=l, histtype="step", linewidth=2, linestyle=linestyle)
    combined += c
    
for c, l in zip(Scovs[5:], labels[5:]):
    _ = hist_dropedges(centers, bins, (np.sqrt(np.diag(c))/SCV), label=l, histtype="step", linewidth=2, linestyle="--")
    combined += c

hist_dropedges(centers, bins, (np.sqrt(np.diag(combined))/SCV), label="Total", histtype="step", color="black", linewidth=2, linestyle=":")

add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="SBND", ylabel="Fractional Uncertainty", 
          legend_loc="upper center", legend_ncol=3, legend_title="Uncorrelated Uncertainties", legend_fontsize=10)

plt.ylim([0, 0.125])

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/SBND_signalbox_systematics_uncorr.png", bbox_inches="tight")
    plt.savefig(PLOTDIR + "/pdf/SBND_signalbox_systematics_uncorr.pdf", bbox_inches="tight")

In [ ]:
combined = np.zeros(Scovs[0].shape)
for c, l in zip(S_detsyst_covs, SDETVAR_NAMES + SCHI2_DETVAR_NAMES + S_TRIGGER_NAMES + ["Corsika", "MC Stat"]):
    if l in (SDETVAR_NAMES + S_TRIGGER_NAMES + ["Corsika"]):
        _ = hist_dropedges(centers, bins, (np.sqrt(np.diag(c))/SCV), label=l, histtype="step", linewidth=2)
        combined += c

hist_dropedges(centers, bins, (np.sqrt(np.diag(combined))/SCV), label="Total", histtype="step", color="black", linewidth=2, linestyle=":")

add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="SBND", ylabel="Fractional Uncertainty", 
          legend_loc="upper center", legend_ncol=3, legend_title="Det. Var. Uncertainties", legend_fontsize=10)

plt.ylim([0, 0.125])

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/SBND_signalbox_systematics_detvar.png", bbox_inches="tight")
    plt.savefig(PLOTDIR + "/pdf/SBND_signalbox_systematics_detvar.pdf", bbox_inches="tight")

In [ ]:
combined = np.zeros(Scovs[0].shape)
for c, l in zip(S_detsyst_covs, SDETVAR_NAMES + SCHI2_DETVAR_NAMES + S_TRIGGER_NAMES + ["Corsika", "MC Stat"]):
    if l in SCHI2_DETVAR_NAMES:
        _ = hist_dropedges(centers, bins, (np.sqrt(np.diag(c))/SCV), label=l, histtype="step", linewidth=2)
        combined += c

hist_dropedges(centers, bins, (np.sqrt(np.diag(combined))/SCV), label="Total", histtype="step", color="black", linewidth=2, linestyle=":")

add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="SBND", ylabel="Fractional Uncertainty", 
          legend_loc="upper center", legend_ncol=3, legend_title="Calorimetric Uncertainties", legend_fontsize=10)

plt.ylim([0, 0.125])

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/SBND_signalbox_systematics_calo.png", bbox_inches="tight")
    plt.savefig(PLOTDIR + "/pdf/SBND_signalbox_systematics_calo.pdf", bbox_inches="tight")

In [ ]:
combined = np.zeros(Icovs[0].shape)
for c, l in zip(I_detsyst_covs, IDETVAR_NAMES + ICHI2_DETVAR_NAMES + I_TRIGGER_NAMES + I_TRACKSPLIT_NAMES + ["MC Stat"]):
    _ = hist_dropedges(centers, bins, (np.sqrt(np.diag(c))/ICV), label=l, histtype="step", linewidth=2)
    combined += c
    
for c, l in zip(Icovs[5:], labels[5:]):
    _ = hist_dropedges(centers, bins, (np.sqrt(np.diag(c))/ICV), label=l, histtype="step", linewidth=2, linestyle="--")
    combined += c

hist_dropedges(centers, bins, (np.sqrt(np.diag(combined))/ICV), label="Total", histtype="step", color="black", linewidth=2, linestyle=":")

add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="ICARUS", ylabel="Fractional Uncertainty", 
          legend_loc="upper center", legend_ncol=3, legend_title="Uncorrelated Uncertainties", legend_fontsize=10)

plt.ylim([0, 0.125])

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/ICARUS_signalbox_systematics_uncorr.png", bbox_inches="tight")
    plt.savefig(PLOTDIR + "/pdf/ICARUS_signalbox_systematics_uncorr.pdf", bbox_inches="tight")

In [ ]:
combined = np.zeros(Icovs[0].shape)
for c, l in zip(I_detsyst_covs, IDETVAR_NAMES + ICHI2_DETVAR_NAMES + I_TRIGGER_NAMES + I_TRACKSPLIT_NAMES + ["MC Stat"]):
    if l in (IDETVAR_NAMES + I_TRIGGER_NAMES + I_TRACKSPLIT_NAMES):
        _ = hist_dropedges(centers, bins, (np.sqrt(np.diag(c))/ICV), label=l, histtype="step", linewidth=2)
        combined += c

hist_dropedges(centers, bins, (np.sqrt(np.diag(combined))/ICV), label="Total", histtype="step", color="black", linewidth=2, linestyle=":")

add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="ICARUS", ylabel="Fractional Uncertainty", 
          legend_loc="upper center", legend_ncol=3, legend_title="Det. Var. Uncertainties", legend_fontsize=10)

plt.ylim([0, 0.125])

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/ICARUS_signalbox_systematics_detvar.png", bbox_inches="tight")
    plt.savefig(PLOTDIR + "/pdf/ICARUS_signalbox_systematics_detvar.pdf", bbox_inches="tight")

In [ ]:
combined = np.zeros(Icovs[0].shape)
for c, l in zip(I_detsyst_covs, IDETVAR_NAMES + ICHI2_DETVAR_NAMES + I_TRIGGER_NAMES + I_TRACKSPLIT_NAMES + ["MC Stat"]):
    if l in (ICHI2_DETVAR_NAMES):
        _ = hist_dropedges(centers, bins, (np.sqrt(np.diag(c))/ICV), label=l, histtype="step", linewidth=2)
        combined += c

hist_dropedges(centers, bins, (np.sqrt(np.diag(combined))/ICV), label="Total", histtype="step", color="black", linewidth=2, linestyle=":")

add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="ICARUS", ylabel="Fractional Uncertainty", 
          legend_loc="upper center", legend_ncol=3, legend_title="Calorimetric Uncertainties", legend_fontsize=10)

plt.ylim([0, 0.125])

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/ICARUS_signalbox_systematics_calo.png", bbox_inches="tight")
    plt.savefig(PLOTDIR + "/pdf/ICARUS_signalbox_systematics_calo.pdf", bbox_inches="tight")

In [ ]:
# drop the lo/hi catch-all bins (as elsewhere), then mask empty-CV bins for the log scale
_sel = ICV[EDGE] > 0
for c, l in zip(Icovs, labels):
    _ = plt.hist(centers[EDGE][_sel], bins=bins[EDGE], weights=(np.sqrt(np.diag(c))/ICV)[EDGE][_sel], label=l, histtype="step", linewidth=2)

add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="ICARUS", ylabel="Fractional Uncertainty", 
          legend_loc="upper center", legend_ncol=2)

plt.ylim([1e-3, 8])

plt.yscale("log")

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/ICARUS_signalbox_systematics_log.png", bbox_inches="tight")
    plt.savefig(PLOTDIR + "/pdf/ICARUS_signalbox_systematics_log.pdf", bbox_inches="tight")

In [ ]:
covs = [s.cov(var, cut, bins, np.concatenate((SCV, ICV))) for s in systematics]
covs.append(np.diag(np.concatenate((SCV, ICV))))

In [ ]:
def corr_f(cov):
    err = np.sqrt(np.diag(cov))
    err[err==0] = 1
    err_inv = np.diag(1/err)
    return err_inv@cov@err_inv

In [ ]:
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib as mpl

In [ ]:
mpl.__version__

In [ ]:
fig, axs = plt.subplots(2, 2, layout='constrained',
    gridspec_kw={'wspace': 0.0, 'hspace': 0.0})
# plt.subplots_adjust(hspace=0, wspace=0)
fig.get_layout_engine().set(w_pad=0 / 72, h_pad=0 / 72, hspace=0,
                            wspace=0)

C = corr_f(covs[0])

nbin = len(bins) - 1
for i, ax in enumerate(axs):
    for j, a in enumerate(ax):
        ic = 1 if i == 0 else 0
        cov = C[nbin*ic:nbin*(ic+1), nbin*j:nbin*(j+1)]
        pcol = a.pcolormesh(bins, bins, cov, cmap="bwr", vmin=-1, vmax=1, linewidth=0, rasterized=True)
        
        a.tick_params(axis='both', which='major', labelsize=FONTSIZE-2)
        if i == 0 and j == 0:
            _ = a.set_xticks([])
            _ = a.set_yticks(np.linspace(bins[0], bins[-1], 7)[1:])
            a.set_ylabel("ICARUS", size=FONTSIZE-2)
        if i == 0 and j == 1:
            _ = a.set_xticks([])
            _ = a.set_yticks([])
        if i == 1 and j == 0:
            _ = a.set_xticks(np.linspace(bins[0], bins[-1], 7)[:-1])
            _ = a.set_yticks(np.linspace(bins[0], bins[-1], 7)[:-1])
            _ = a.set_ylabel("SBND", size=FONTSIZE-2)
            _ = a.set_xlabel("SBND", size=FONTSIZE-2)
        elif i == 1 and j == 1:
            pass
            _ = a.set_xticks(np.linspace(bins[0], bins[-1], 7)[1:])
            _ = a.set_yticks([])
            _ = a.set_xlabel("ICARUS", size=FONTSIZE-2)


_ = fig.suptitle("Flux Systematic Correlation", size=FONTSIZE+2) #, x=0.46, y=0.95)
_ = fig.supxlabel("Reco. Neutrino Energy [GeV]", size=FONTSIZE)
_ = fig.supylabel("Reco. Neutrino Energy [GeV]", size=FONTSIZE)

norm = mcolors.Normalize(vmin=-1, vmax=1)
sm = cm.ScalarMappable(cmap="bwr", norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=[a for ax in axs for a in ax])
cbar.set_label("Correlation", rotation=270, size=FONTSIZE-2, labelpad=12)

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/flux_correlation.png")
    plt.savefig(PLOTDIR + "/pdf/flux_correlation.pdf")

In [ ]:
fig, axs = plt.subplots(2, 2, layout='constrained',
    gridspec_kw={'wspace': 0.0, 'hspace': 0.0})
# plt.subplots_adjust(hspace=0, wspace=0)
fig.get_layout_engine().set(w_pad=0 / 72, h_pad=0 / 72, hspace=0,
                            wspace=0)

C = corr_f(covs[1])

nbin = len(bins) - 1
for i, ax in enumerate(axs):
    for j, a in enumerate(ax):
        ic = 1 if i == 0 else 0
        cov = C[nbin*ic:nbin*(ic+1), nbin*j:nbin*(j+1)]
        pcol = a.pcolormesh(bins, bins, cov, cmap="bwr", vmin=-1, vmax=1, linewidth=0, rasterized=True)

        a.tick_params(axis='both', which='major', labelsize=FONTSIZE-2)
        if i == 0 and j == 0:
            _ = a.set_xticks([])
            _ = a.set_yticks(np.linspace(bins[0], bins[-1], 7)[1:])
            _ = a.set_ylabel("ICARUS", size=FONTSIZE-2)
        if i == 0 and j == 1:
            _ = a.set_xticks([])
            _ = a.set_yticks([])
        if i == 1 and j == 0:
            _ = a.set_xticks(np.linspace(bins[0], bins[-1], 7)[:-1])
            _ = a.set_yticks(np.linspace(bins[0], bins[-1], 7)[:-1])
            _ = a.set_ylabel("SBND", size=FONTSIZE-2)
            _ = a.set_xlabel("SBND", size=FONTSIZE-2)
        elif i == 1 and j == 1:
            pass
            _ = a.set_xticks(np.linspace(bins[0], bins[-1], 7)[1:])
            _ = a.set_yticks([])
            _ = a.set_xlabel("ICARUS", size=FONTSIZE-2)


_ = fig.suptitle("Geant4 Systematic Correlation", size=FONTSIZE+2) #, x=0.46, y=0.95)
_ = fig.supxlabel("Reco. Neutrino Energy [GeV]", size=FONTSIZE)
_ = fig.supylabel("Reco. Neutrino Energy [GeV]", size=FONTSIZE)

norm = mcolors.Normalize(vmin=-1, vmax=1)
sm = cm.ScalarMappable(cmap="bwr", norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=[a for ax in axs for a in ax])
cbar.set_label("Correlation", rotation=270, size=FONTSIZE-2, labelpad=12)

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/g4_correlation.png")
    plt.savefig(PLOTDIR + "/pdf/g4_correlation.pdf")

In [ ]:
fig, axs = plt.subplots(2, 2, layout='constrained',
    gridspec_kw={'wspace': 0.0, 'hspace': 0.0})
# plt.subplots_adjust(hspace=0, wspace=0)
fig.get_layout_engine().set(w_pad=0 / 72, h_pad=0 / 72, hspace=0,
                            wspace=0)

C = corr_f(covs[2])

nbin = len(bins) - 1
for i, ax in enumerate(axs):
    for j, a in enumerate(ax):
        ic = 1 if i == 0 else 0
        cov = C[nbin*ic:nbin*(ic+1), nbin*j:nbin*(j+1)]
        pcol = a.pcolormesh(bins, bins, cov, cmap="bwr", vmin=-1, vmax=1, linewidth=0, rasterized=True)

        a.tick_params(axis='both', which='major', labelsize=FONTSIZE-2)
        if i == 0 and j == 0:
            _ = a.set_xticks([])
            _ = a.set_yticks(np.linspace(bins[0], bins[-1], 7)[1:])
            _ = a.set_ylabel("ICARUS", size=FONTSIZE-2)
        if i == 0 and j == 1:
            _ = a.set_xticks([])
            _ = a.set_yticks([])
        if i == 1 and j == 0:
            _ = a.set_xticks(np.linspace(bins[0], bins[-1], 7)[:-1])
            _ = a.set_yticks(np.linspace(bins[0], bins[-1], 7)[:-1])
            _ = a.set_ylabel("SBND", size=FONTSIZE-2)
            _ = a.set_xlabel("SBND", size=FONTSIZE-2)
        elif i == 1 and j == 1:
            pass
            _ = a.set_xticks(np.linspace(bins[0], bins[-1], 7)[1:])
            _ = a.set_yticks([])
            _ = a.set_xlabel("ICARUS", size=FONTSIZE-2)


_ = fig.suptitle("XSec Systematic Correlation", size=FONTSIZE+2) #, x=0.46, y=0.95)
_ = fig.supxlabel("Reco. Neutrino Energy [GeV]", size=FONTSIZE)
_ = fig.supylabel("Reco. Neutrino Energy [GeV]", size=FONTSIZE)

norm = mcolors.Normalize(vmin=-1, vmax=1)
sm = cm.ScalarMappable(cmap="bwr", norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=[a for ax in axs for a in ax])
cbar.set_label("Correlation", rotation=270, size=FONTSIZE-2, labelpad=12)

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/xsec_correlation.png")
    plt.savefig(PLOTDIR + "/pdf/xsec_correlation.pdf")

In [ ]:
plt.title("XSec Systematics")

plt.imshow(corr_f(covs[2]), origin="lower", vmin=0.8, vmax=1)
plt.colorbar()

In [ ]:
fig, axs = plt.subplots(2, 2, layout='constrained',
    gridspec_kw={'wspace': 0.0, 'hspace': 0.0})
# plt.subplots_adjust(hspace=0, wspace=0)
fig.get_layout_engine().set(w_pad=0 / 72, h_pad=0 / 72, hspace=0,
                            wspace=0)

C = corr_f(covs[3])

nbin = len(bins) - 1
for i, ax in enumerate(axs):
    for j, a in enumerate(ax):
        ic = 1 if i == 0 else 0
        cov = C[nbin*ic:nbin*(ic+1), nbin*j:nbin*(j+1)]
        pcol = a.pcolormesh(bins, bins, cov, cmap="bwr", vmin=-1, vmax=1, linewidth=0, rasterized=True)

        a.tick_params(axis='both', which='major', labelsize=FONTSIZE-2)
        if i == 0 and j == 0:
            _ = a.set_xticks([])
            _ = a.set_yticks(np.linspace(bins[0], bins[-1], 7)[1:])
            _ = a.set_ylabel("ICARUS", size=FONTSIZE-2)
        if i == 0 and j == 1:
            _ = a.set_xticks([])
            _ = a.set_yticks([])
        if i == 1 and j == 0:
            _ = a.set_xticks(np.linspace(bins[0], bins[-1], 7)[:-1])
            _ = a.set_yticks(np.linspace(bins[0], bins[-1], 7)[:-1])
            _ = a.set_ylabel("SBND", size=FONTSIZE-2)
            _ = a.set_xlabel("SBND", size=FONTSIZE-2)
        elif i == 1 and j == 1:
            pass
            _ = a.set_xticks(np.linspace(bins[0], bins[-1], 7)[1:])
            _ = a.set_yticks([])
            _ = a.set_xlabel("ICARUS", size=FONTSIZE-2)


_ = fig.suptitle("POT Norm. Systematic Correlation", size=FONTSIZE+2) #, x=0.46, y=0.95)
_ = fig.supxlabel("Reco. Neutrino Energy [GeV]", size=FONTSIZE)
_ = fig.supylabel("Reco. Neutrino Energy [GeV]", size=FONTSIZE)

norm = mcolors.Normalize(vmin=-1, vmax=1)
sm = cm.ScalarMappable(cmap="bwr", norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=[a for ax in axs for a in ax])
cbar.set_label("Correlation", rotation=270, size=FONTSIZE-2, labelpad=12)

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/norm_correlation.png")
    plt.savefig(PLOTDIR + "/pdf/norm_correlation.pdf")

In [ ]:
fig, axs = plt.subplots(2, 2, layout='constrained',
    gridspec_kw={'wspace': 0.0, 'hspace': 0.0})
# plt.subplots_adjust(hspace=0, wspace=0)
fig.get_layout_engine().set(w_pad=0 / 72, h_pad=0 / 72, hspace=0,
                            wspace=0)

C = corr_f(covs[4])

nbin = len(bins) - 1
for i, ax in enumerate(axs):
    for j, a in enumerate(ax):
        ic = 1 if i == 0 else 0
        cov = C[nbin*ic:nbin*(ic+1), nbin*j:nbin*(j+1)]
        pcol = a.pcolormesh(bins, bins, cov, cmap="bwr", vmin=-1, vmax=1, linewidth=0, rasterized=True)

        a.tick_params(axis='both', which='major', labelsize=FONTSIZE-2)
        if i == 0 and j == 0:
            _ = a.set_xticks([])
            _ = a.set_yticks(np.linspace(bins[0], bins[-1], 7)[1:])
            _ = a.set_ylabel("ICARUS", size=FONTSIZE-2)
        if i == 0 and j == 1:
            _ = a.set_xticks([])
            _ = a.set_yticks([])
        if i == 1 and j == 0:
            _ = a.set_xticks(np.linspace(bins[0], bins[-1], 7)[:-1])
            _ = a.set_yticks(np.linspace(bins[0], bins[-1], 7)[:-1])
            _ = a.set_ylabel("SBND", size=FONTSIZE-2)
            _ = a.set_xlabel("SBND", size=FONTSIZE-2)
        elif i == 1 and j == 1:
            pass
            _ = a.set_xticks(np.linspace(bins[0], bins[-1], 7)[1:])
            _ = a.set_yticks([])
            _ = a.set_xlabel("ICARUS", size=FONTSIZE-2)


_ = fig.suptitle("Detector Systematic Correlation", size=FONTSIZE+2) #, x=0.46, y=0.95)
_ = fig.supxlabel("Reco. Neutrino Energy [GeV]", size=FONTSIZE)
_ = fig.supylabel("Reco. Neutrino Energy [GeV]", size=FONTSIZE)

norm = mcolors.Normalize(vmin=-1, vmax=1)
sm = cm.ScalarMappable(cmap="bwr", norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=[a for ax in axs for a in ax])
cbar.set_label("Correlation", rotation=270, size=FONTSIZE-2, labelpad=12)

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/detector_correlation.png")
    plt.savefig(PLOTDIR + "/pdf/detector_correlation.pdf")

In [ ]:
def ratio_cov_full(x, y, cov):
    """
    Covariance of r = x / y given the full covariance of (x, y).

    Parameters
    ----------
    x, y : array-like, shape (n,)
        Central values
    cov : array-like, shape (2n, 2n)
        Full covariance matrix of (x, y)

        Ordering must be:
        cov = [[Cov(x,x), Cov(x,y)],
               [Cov(y,x), Cov(y,y)]]

    Returns
    -------
    cov_r : ndarray, shape (n, n)
        Covariance matrix of r
    """
    n = len(x)
    assert cov.shape == (2*n, 2*n)

    # Protect against division by zero
    eps = 1e-12
    y_safe = np.where(np.abs(y) < eps, eps, y)

    Dx = np.diag(1.0 / y_safe)
    Dy = np.diag(-x / y_safe**2)

    # Full Jacobian: shape (n, 2n)
    J = np.hstack([Dx, Dy])

    return J @ cov @ J.T

In [ ]:
def conditional_constraint_full(cov):
    """
    Conditional (Gaussian) constraint on y given a measurement of x,
    using the full joint covariance of (x, y).

        cov:   Cov(y,y) - Cov(y,x) Cov(x,x)^{-1} Cov(x,y)

    i.e. the standard Schur-complement / near-detector-style conditional
    constraint. The conditional covariance depends only on the covariance
    blocks, not on x, y, or x_obs.

    Parameters
    ----------
    cov : array-like, shape (2n, 2n)
        Full covariance matrix of (x, y).
        Ordering must be:
        cov = [[Cov(x,x), Cov(x,y)],
               [Cov(y,x), Cov(y,y)]]

    Returns
    -------
    cov_cond : ndarray, shape (n, n)
        Conditional (post-constraint) covariance of y.
    """

    cov = np.asarray(cov, dtype=float)
    n = cov.shape[0]//2
    assert cov.shape == (2*n, 2*n)


    # Block decomposition
    Cxx = cov[:n, :n]
    Cxy = cov[:n, n:]
    Cyx = cov[n:, :n]
    Cyy = cov[n:, n:]

    # Gain matrix K = Cyx Cxx^{-1}
    K = Cyx @ np.linalg.inv(Cxx)

    cov_cond = Cyy - K @ Cxy
    cov_cond = 0.5 * (cov_cond + cov_cond.T)  # enforce symmetry vs. roundoff
    return cov_cond

In [ ]:
labels = [
    "Flux",
    "G4",
    "XSec",
    "POT Norm.",
    "Detector",
    "Dirt",
    "Beam Off",
    "Stat",
]

In [ ]:
len(covs)

In [ ]:
ratio = SCV / ICV
# ratio = ratio / ratio.mean()

ratio_cov_flux = ratio_cov_full(SCV, ICV, covs[0])
ratio_cov_g4 = ratio_cov_full(SCV, ICV, covs[1])
ratio_cov_xsec = ratio_cov_full(SCV, ICV, covs[2])
ratio_cov_norm = ratio_cov_full(SCV, ICV, covs[3])
ratio_cov_det = ratio_cov_full(SCV, ICV, covs[4] + covs[5] + covs[6])
ratio_cov_stat = ratio_cov_full(SCV, ICV, covs[7])
ratio_cov_all = ratio_cov_full(SCV, ICV, np.sum(covs, axis=0))

In [ ]:
Sdf.crthit.mean()

In [ ]:
_ = hist_dropedges(centers, bins, np.sqrt(np.diag(ratio_cov_flux))/ratio, histtype="step", linewidth=2, label="Flux")
_ = hist_dropedges(centers, bins, np.sqrt(np.diag(ratio_cov_g4))/ratio, histtype="step", linewidth=2, label="G4")
_ = hist_dropedges(centers, bins, np.sqrt(np.diag(ratio_cov_xsec))/ratio, histtype="step", linewidth=2, label="XSec")
_ = hist_dropedges(centers, bins, np.sqrt(np.diag(ratio_cov_det))/ratio, histtype="step", linewidth=2, label="Detector")
_ = hist_dropedges(centers, bins, np.sqrt(np.diag(ratio_cov_norm))/ratio, histtype="step", linewidth=2, label="POT Norm.")
# _ = hist_dropedges(centers, bins, np.sqrt(np.diag(ratio_cov_dirt))/ratio, histtype="step", linewidth=2, label="Dirt")
# _ = hist_dropedges(centers, bins, np.sqrt(np.diag(ratio_cov_beamoff))/ratio, histtype="step", linewidth=2, label="Beam-Off")
_ = hist_dropedges(centers, bins, np.sqrt(np.diag(ratio_cov_stat))/ratio, histtype="step", linewidth=2, label="Stat.")
_ = hist_dropedges(centers, bins, np.sqrt(np.diag(ratio_cov_all))/ratio, histtype="step", linewidth=2, label="All")
# plt.ylim([0, 0.25])

add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="SBND / ICARUS", 
          ylabel="Uncertainty on SBND/ICARUS Ratio", legend_loc="upper left", legend_ncol=2)

plt.ylim([0, 0.2])

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/ratio_signalbox_systematics.png", bbox_inches="tight")
    plt.savefig(PLOTDIR + "/pdf/ratio_signalbox_systematics.pdf", bbox_inches="tight")

In [ ]:
cond_cov_flux = conditional_constraint_full(np.sum(covs, axis=0) - covs[0])
cond_cov_g4 = conditional_constraint_full(np.sum(covs, axis=0) - covs[1])
cond_cov_xsec = conditional_constraint_full(np.sum(covs, axis=0) - covs[2])
cond_cov_norm = conditional_constraint_full(np.sum(covs, axis=0) - covs[3])
cond_cov_det = conditional_constraint_full(np.sum(covs, axis=0) - (covs[4] + covs[5] + covs[6]))
cond_cov_stat = conditional_constraint_full(np.sum(covs, axis=0) - covs[7])
cond_cov_all = conditional_constraint_full(np.sum(covs, axis=0))

In [ ]:
_ = hist_dropedges(centers, bins, np.sqrt(np.diag(cond_cov_flux))/ICV, histtype="step", linewidth=2, label="No Flux")
_ = hist_dropedges(centers, bins, np.sqrt(np.diag(cond_cov_g4))/ICV, histtype="step", linewidth=2, label="No G4")
_ = hist_dropedges(centers, bins, np.sqrt(np.diag(cond_cov_xsec))/ICV, histtype="step", linewidth=2, label="No XSec")
_ = hist_dropedges(centers, bins, np.sqrt(np.diag(cond_cov_det))/ICV, histtype="step", linewidth=2, label="No Detector")
_ = hist_dropedges(centers, bins, np.sqrt(np.diag(cond_cov_norm))/ICV, histtype="step", linewidth=2, label="No POT")
_ = hist_dropedges(centers, bins, np.sqrt(np.diag(cond_cov_stat))/ICV, histtype="step", linewidth=2, label="No Stat.")
_ = hist_dropedges(centers, bins, np.sqrt(np.diag(cond_cov_all))/ICV, histtype="step", linewidth=2, label="All")
# plt.ylim([0, 0.25])

add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="", 
          ylabel="Frac. Unc. on ICARUS\nConditioned by SBND", legend_loc="upper left", legend_ncol=2)

# plt.ylim([0, 0.15])

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/condconstraint_signalbox_systematics.png", bbox_inches="tight")
    plt.savefig(PLOTDIR + "/pdf/condconstraint_signalbox_systematics.pdf", bbox_inches="tight")

In [ ]:
_ = plt.hist(centers, bins=bins, weights=ratio, histtype="step", linewidth=2)
add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="SBND / ICARUS", ylabel="SBND/ICARUS Ratio")
plt.axhline([1], color="black", linestyle="--")

In [ ]:
# Per-knob xsec breakdown: each weight-based knob plus the binding-energy
# shift, each fully correlated SBND<->ICARUS like the total.
xs = SXSecWgtSyst
xsec_singles = [(w, syst.CorrelatedSystematic(syst.WeightSystematic(Sdf, [w], avg=False), syst.WeightSystematic(Idf, [w], avg=False)))
                for w in xs.wgts]
xsec_singles.append(("BindingEnergy", syst.CorrelatedSystematic(S_BE_syst, I_BE_syst)))
xsec_names = [w for w, _ in xsec_singles]

xsec_unc_ratios = []
xsec_unc_sbnd = []
xsec_unc_icarus = []

for w, s in xsec_singles:
    cov = s.cov(var, cut, bins, np.concatenate((SCV, ICV)))
    ratio_unc = ratio_cov_full(SCV, ICV, cov)
    xsec_unc_ratios.append(np.sqrt(np.diag(ratio_unc))/ratio)
    xsec_unc_sbnd.append(np.sqrt(np.diag(s.systa.cov(var, cut, bins, SCV))) / SCV)
    xsec_unc_icarus.append(np.sqrt(np.diag(s.systb.cov(var, cut, bins, ICV))) / ICV)


In [ ]:
xsec_unc_ratios[0].shape

In [ ]:
np.max(xsec_unc_sbnd, axis=1).shape, np.max(xsec_unc_ratios, axis=1).shape

In [ ]:
plt.scatter(np.sqrt(np.mean(np.array(xsec_unc_sbnd)**2, axis=1)), np.sqrt(np.mean(np.array(xsec_unc_ratios)**2, axis=1)))
plt.plot([0, 0.1], [0, 0.1], color="red")

plt.ylabel("RMS Uncertainty on SBND/ICARUS")
plt.xlabel("RMS Uncertainty on SBND")
plt.title("Cross Seciton Uncertainties")

In [ ]:
def weightname(w):
    if w == "BindingEnergy":
        return "Binding Energy"
    w = w.split("_")
    if w[0] == "CCQETemplateReweight":
        return w[3] + " " + w[4]
    elif w[0] == "QEInterference":
        return w[3] + " bin" + w[5]
    elif w[0] == "GENIEReWeight":
        return " ".join(w[3:-1])
    elif w[0] == "ZExpPCAWeighter":
        return "axialFF bin" + w[4][-1] 
    elif w[0] == "MECq0q3InterpWeighting":
        return "MEC %s %s" % (w[3], w[5][-1])
    else:
        print(w)
        assert(False)

In [ ]:
ifig = 0

for r, rS, rI, w in zip(xsec_unc_ratios, xsec_unc_sbnd, xsec_unc_icarus, xsec_names):
    if np.any(r > 0.03):
        _ = plt.figure(ifig)
        _ = plt.hist(centers, bins=bins, weights=rS, histtype="step", linewidth=2, label="SBND")
        _ = plt.hist(centers, bins=bins, weights=rI, histtype="step", linewidth=2, label="ICAURS")
        _ = plt.hist(centers, bins=bins, weights=r, histtype="step", linewidth=2, label="Ratio")

        _ = plt.legend()
        _ = plt.title(w)

        ifig += 1
        add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det=weightname(w), 
          ylabel="Frac. Unc. on ICARUS\nConditioned by SBND")

In [ ]:
xsec_unc_constraint = []
xsec_unc_single_constraint = []

for w, s in xsec_singles:
    cov = s.cov(var, cut, bins, np.concatenate((SCV, ICV)))
    constraint = conditional_constraint_full(np.sum(covs, axis=0) - cov)
    xsec_unc_constraint.append(constraint)
    xsec_unc_single_constraint.append(conditional_constraint_full(cov + covs[7]))

stat_unc_constraint = conditional_constraint_full(covs[7])


In [ ]:
_ = plt.hist(centers, bins=bins, weights=np.sqrt(np.diag(cond_cov_all))/ICV, histtype="step", linewidth=2, label="All")

for w,c,r in zip(xsec_names, xsec_unc_constraint, xsec_unc_ratios):
    if np.any(r > 0.03):
        _ = plt.hist(centers, bins=bins, weights=np.sqrt(np.diag(c))/ICV, histtype="step", linewidth=2, label="Without %s" % weightname(w))

add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="", 
          ylabel="Frac. Unc. on ICARUS\nConditioned by SBND", legend_loc="upper left", legend_ncol=1, legend_fontsize=12)

# plt.ylim([0, 0.15])

In [ ]:
ifig = 0

for r, sc, rS, rI, w in zip(xsec_unc_ratios, xsec_unc_single_constraint, xsec_unc_sbnd, xsec_unc_icarus, xsec_names):
    if np.any(r > 0.03):
        _ = plt.figure(ifig)
        _ = plt.hist(centers, bins=bins, weights=rS, histtype="step", linewidth=2, label="SBND")
        _ = plt.hist(centers, bins=bins, weights=rI, histtype="step", linewidth=2, label="ICAURS")
        _ = plt.hist(centers, bins=bins, weights=np.sqrt(np.diag(stat_unc_constraint))/ICV, histtype="step", linewidth=2, label="Conditioned ICARUS Stat")
        _ = plt.hist(centers, bins=bins, weights=np.sqrt(np.diag(sc))/ICV, histtype="step", linewidth=2, label="Conditioned ICARUS Stat+Syst")

        _ = plt.legend()
        _ = plt.title(w)

        ifig += 1

In [ ]:
for r,w in zip(xsec_unc_ratios, xsec_names):
    
    if np.any(r > 0.02):
        _ = plt.hist(centers, bins=bins, weights=r, histtype="step", linewidth=2, label=weightname(w))

_ = plt.hist(centers, bins=bins, weights=np.sqrt(np.diag(ratio_cov_xsec))/ratio, histtype="step", linewidth=2, label="Total XSec", color="black")

add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="SBND / ICARUS", 
          ylabel="Uncertainty on SBND/ICARUS Ratio", legend_loc="upper left", legend_ncol=2, legend_fontsize=11)

In [ ]:
for w in xs.wgts:
    if "CCQETemplate" in w or "QEInt" in w:
        Sdf[w + "_clipped"] = np.clip(Sdf[w], 0, 3)
        Idf[w + "_clipped"] = np.clip(Idf[w], 0, 3)

In [ ]:
xs = SXSecWgtSyst
xsec_unc_ratios_clipped = []

for w in xs.wgts:
    if "CCQETemplate" in w or "QEInt" in w:
        s = syst.CorrelatedSystematic(syst.WeightSystematic(Sdf, [w + "_clipped"], avg=False), syst.WeightSystematic(Idf, [w + "_clipped"], avg=False))
        cov = s.cov(var, cut, bins, np.concatenate((SCV, ICV)))
        ratio_unc = ratio_cov_full(SCV, ICV, cov)
        xsec_unc_ratios_clipped.append(ratio_unc)

In [ ]:
ifig = 0
for r,w in zip(xsec_unc_ratios_clipped, xs.wgts):
    
    if "CCQETemplate" in w or "QEInt" in w:
        _ = plt.figure(ifig)
        _ = plt.hist(centers, bins=bins, weights=np.sqrt(np.diag(r))/ratio, histtype="step", linewidth=2, label="Clipped 10")
        _ = plt.hist(centers, bins=bins, weights=np.sqrt(np.diag(xsec_unc_ratios_clipped[ifig]))/ratio, histtype="step", linewidth=2, label="Clipped 3")

        _ = plt.title(w)
        _ = plt.legend()
    ifig += 1
    add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det=weightname(w), 
          ylabel="Uncertainty on SBND/ICARUS Ratio")

In [ ]:
reco_bins = bins
true_bins = np.array([-2] + list(reco_bins) + [np.inf])
true_bins_centers = (true_bins[:-1] + true_bins[1:])/2
true_bins_centers, true_bins

In [ ]:
SCV2 = np.histogramdd([Sdf.loc[Sdf[cut], "nu_E_calo"], Sdf.loc[Sdf[cut], "true_E"]], 
                      bins=[reco_bins, true_bins], weights=Sdf.loc[Sdf[cut], wgt])[0]

ICV2 = np.histogramdd([Idf.loc[Idf[cut], "nu_E_calo"], Idf.loc[Idf[cut], "true_E"]], 
                      bins=[reco_bins, true_bins], weights=Idf.loc[Idf[cut], wgt])[0]

SCVT = np.histogram(Sdf.loc[Sdf[cut], "true_E"], bins=true_bins, weights=Sdf.loc[Sdf[cut], wgt])[0]
ICVT = np.histogram(Idf.loc[Idf[cut], "true_E"], bins=true_bins, weights=Idf.loc[Idf[cut], wgt])[0]

# CV2D = np.concatenate((SCV2.T, ICV2.T), axis=0) # .flatten()
# CV2D = np.concatenate((CV2D, CV2D), axis=1)

Imig = ICV2.T / ICV2.sum(axis=1)
Smig = SCV2.T / SCV2.sum(axis=1)

migration = np.block([
    [Smig, np.zeros(Smig.shape)],
    [np.zeros(Imig.shape), Imig]
])

def unfold(C):
    return migration@C@migration.T

def unfoldSBND(C):
    return Smig@C@Smig.T

def unfoldICARUS(C):
    return Imig@C@Imig.T

In [ ]:
plt.imshow(migration, origin="lower")

In [ ]:
ratioT = SCVT / ICVT
ratioT = ratioT / ratioT.mean()

ratioT_cov_flux = ratio_cov_full(SCVT, ICVT, unfold(covs[0]))
ratioT_cov_g4 = ratio_cov_full(SCVT, ICVT, unfold(covs[1]))
ratioT_cov_xsec = ratio_cov_full(SCVT, ICVT, unfold(covs[2]))
ratioT_cov_norm = ratio_cov_full(SCVT, ICVT, unfold(covs[3]))
ratioT_cov_det = ratio_cov_full(SCVT, ICVT, unfold(covs[4] + covs[5] + covs[6]))
ratioT_cov_stat = ratio_cov_full(SCVT, ICVT, unfold(covs[7]))
ratioT_cov_all = ratio_cov_full(SCVT, ICVT, unfold(np.sum(covs, axis=0)))

In [ ]:
plot_bins = true_bins[1:-1]

_ = plt.hist(true_bins_centers[1:-1], bins=plot_bins, weights=(np.sqrt(np.diag(ratioT_cov_flux))/ratioT)[1:-1],
             histtype="step", linewidth=2, label="Flux")
_ = plt.hist(true_bins_centers[1:-1], bins=plot_bins, weights=(np.sqrt(np.diag(ratioT_cov_g4))/ratioT)[1:-1],
             histtype="step", linewidth=2, label="G4")
_ = plt.hist(true_bins_centers[1:-1], bins=plot_bins, weights=(np.sqrt(np.diag(ratioT_cov_xsec))/ratioT)[1:-1],
             histtype="step", linewidth=2, label="XSec")
_ = plt.hist(true_bins_centers[1:-1], bins=plot_bins, weights=(np.sqrt(np.diag(ratioT_cov_det))/ratioT)[1:-1],
             histtype="step", linewidth=2, label="Detector")
_ = plt.hist(true_bins_centers[1:-1], bins=plot_bins, weights=(np.sqrt(np.diag(ratioT_cov_norm))/ratioT)[1:-1],
             histtype="step", linewidth=2, label="POT Norm.")
_ = plt.hist(true_bins_centers[1:-1], bins=plot_bins, weights=(np.sqrt(np.diag(ratioT_cov_stat))/ratioT)[1:-1],
             histtype="step", linewidth=2, label="Stat.")
_ = plt.hist(true_bins_centers[1:-1], bins=plot_bins, weights=(np.sqrt(np.diag(ratioT_cov_all))/ratioT)[1:-1],
             histtype="step", linewidth=2, label="All")
# plt.ylim([0, 0.25])

add_style(plt.gca(), "True Neutrino Energy [GeV]", det="SBND / ICARUS", 
          ylabel="Unfolded Unc. on SBND/ICARUS Ratio", legend_loc="upper center", legend_ncol=2)

plt.ylim([0, 0.15])

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/ratio_signalbox_systematics_trueE.png", bbox_inches="tight")
    plt.savefig(PLOTDIR + "/pdf/ratio_signalbox_systematics_trueE.pdf", bbox_inches="tight")

In [ ]:
combined = np.zeros(Icovs[0].shape)
for c, l in zip(I_detsyst_covs, IDETVAR_NAMES + ICHI2_DETVAR_NAMES + I_TRIGGER_NAMES + I_TRACKSPLIT_NAMES + ["MC Stat"]):
    linestyle = "--" if l == "MC Stat" else "-"
    _ = plt.hist(true_bins_centers[1:-1], bins=plot_bins, weights=(np.sqrt(np.diag(unfoldICARUS(c)))/ICVT)[1:-1], label=l, 
                 histtype="step", linewidth=2, linestyle=linestyle)
    combined += c
    
for c, l in zip(Icovs[5:], labels[5:]):
    _ = plt.hist(true_bins_centers[1:-1], bins=plot_bins, weights=(np.sqrt(np.diag(unfoldICARUS(c)))/ICVT)[1:-1], label=l, 
                 histtype="step", linewidth=2, linestyle="--")
    combined += c

plt.hist(true_bins_centers[1:-1], bins=plot_bins, weights=(np.sqrt(np.diag(unfoldICARUS(combined)))/ICVT)[1:-1], label="Total", 
         histtype="step", color="black", linewidth=2, linestyle=":")

add_style(plt.gca(), "True Neutrino Energy [GeV]", det="ICARUS", ylabel="Unfolded Fractional Uncertainty", 
          legend_loc="upper center", legend_ncol=3, legend_title="Uncorrelated Uncertainties", legend_fontsize=10)

plt.ylim([0, 0.08])

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/ICARUS_signalbox_systematics_trueE_uncorr.png", bbox_inches="tight")
    plt.savefig(PLOTDIR + "/pdf/ICARUS_signalbox_systematics_trueE_uncorr.pdf", bbox_inches="tight")

In [ ]:
combined = np.zeros(Scovs[0].shape)
for c, l in zip(S_detsyst_covs, SDETVAR_NAMES + SCHI2_DETVAR_NAMES + S_TRIGGER_NAMES+ ["Corsika", "MC Stat"]):
    linestyle = "--" if l == "MC Stat" else "-"
    _ = plt.hist(true_bins_centers[1:-1], bins=plot_bins, weights=(np.sqrt(np.diag(unfoldSBND(c)))/SCVT)[1:-1], label=l, histtype="step",
                 linewidth=2, linestyle=linestyle)
    combined += c
    
for c, l in zip(Scovs[5:], labels[5:]):
    _ = plt.hist(true_bins_centers[1:-1], bins=plot_bins, weights=(np.sqrt(np.diag(unfoldSBND(c)))/SCVT)[1:-1], 
                 label=l, histtype="step", linewidth=2, linestyle="--")
    combined += c

plt.hist(true_bins_centers[1:-1], bins=plot_bins, weights=(np.sqrt(np.diag(unfoldSBND(combined)))/SCVT)[1:-1], label="Total", 
         histtype="step", color="black", linewidth=2, linestyle=":")

add_style(plt.gca(), "True Neutrino Energy [GeV]", det="SBND", ylabel="Unfolded Fractional Uncertainty", 
          legend_loc="upper center", legend_ncol=3, legend_title="Uncorrelated Uncertainties", legend_fontsize=10)

plt.ylim([0, 0.08])

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/SBND_signalbox_systematics_trueE_uncorr.png", bbox_inches="tight")
    plt.savefig(PLOTDIR + "/pdf/SBND_signalbox_systematics_trueE_uncorr.pdf", bbox_inches="tight")

In [ ]:
_ = plt.hist(true_bins_centers[1:-1], bins=plot_bins, weights=(np.sqrt(np.diag(unfoldICARUS(cond_cov_flux)))/ICVT)[1:-1], histtype="step", linewidth=2, label="No Flux")
_ = plt.hist(true_bins_centers[1:-1], bins=plot_bins, weights=(np.sqrt(np.diag(unfoldICARUS(cond_cov_g4)))/ICVT)[1:-1], histtype="step", linewidth=2, label="No G4")
_ = plt.hist(true_bins_centers[1:-1], bins=plot_bins, weights=(np.sqrt(np.diag(unfoldICARUS(cond_cov_xsec)))/ICVT)[1:-1], histtype="step", linewidth=2, label="No XSec")
_ = plt.hist(true_bins_centers[1:-1], bins=plot_bins, weights=(np.sqrt(np.diag(unfoldICARUS(cond_cov_det)))/ICVT)[1:-1], histtype="step", linewidth=2, label="No Detector")
_ = plt.hist(true_bins_centers[1:-1], bins=plot_bins, weights=(np.sqrt(np.diag(unfoldICARUS(cond_cov_norm)))/ICVT)[1:-1], histtype="step", linewidth=2, label="No POT")
_ = plt.hist(true_bins_centers[1:-1], bins=plot_bins, weights=(np.sqrt(np.diag(unfoldICARUS(cond_cov_stat)))/ICVT)[1:-1], histtype="step", linewidth=2, label="No Stat.")
_ = plt.hist(true_bins_centers[1:-1], bins=plot_bins, weights=(np.sqrt(np.diag(unfoldICARUS(cond_cov_all)))/ICVT)[1:-1], histtype="step", linewidth=2, label="All")
# plt.ylim([0, 0.25])

add_style(plt.gca(), "True. Neutrino Energy [GeV]", det="", 
          ylabel="Unfolded Frac. Unc. on ICARUS\nConditioned by SBND", legend_loc="upper right", legend_ncol=2)

plt.ylim([0, 0.075])

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/condconstraint_signalbox_systematics_trueE.png", bbox_inches="tight")
    plt.savefig(PLOTDIR + "/pdf/condconstraint_signalbox_systematics_trueE.pdf", bbox_inches="tight")